# Hearo — YAMNet 공정 초기화 반복 검증 v3.2

v3.2는 네 하드 네거티브 방법의 순수한 효과와 학습 초기화 변동을 분리하기 위한 논문 검증용 노트북입니다.

- 각 outer split에서 네 방법 모두 동일한 최종 학습 seed를 사용합니다.
- 각 split을 `split+1000`, `split+2000`, `split+3000`의 세 초기화로 반복합니다.
- 대표 split 42에서는 네 방법 모두 1042·2042·3042를 공유합니다.
- seed 목록, 방법, 임계값은 test를 보기 전에 고정하며 test 결과로 seed를 선택하지 않습니다.
- 배포 TFLite는 사전 지정된 첫 seed(대표 split은 1042)만 사용합니다.
- 방법별·split별·초기화별 결과와 paired 차이를 모두 저장합니다.
- 표준화된 사이렌 라벨(Hi-Lo/Wail/Yelp 의미)을 보존합니다.
- revision이 일치하는 v3 OOF/CV 및 YAMNet 임베딩 캐시는 재사용하지만 v3.2 결과는 별도 폴더에 저장합니다.

데이터 위치 기본값: `/content/drive/MyDrive/소리 정리`  
결과 위치 기본값: `/content/drive/MyDrive/Hearo_model_colab_paper_v3_2_newdata_20260925`

> 대표 분할 한 번의 결과나 유리한 초기화 하나를 골라 제안 방법의 효과로 보고하지 마세요.


## 1. Colab 고속 실행 설정


In [ ]:
# fast: 빠른 반복 실험 / paper: 최종 논문용 전체 검증
RUN_MODE = "paper"

# Drive I/O 병목을 줄이기 위해 음원을 Colab 로컬 SSD로 한 번 복사합니다.
USE_LOCAL_DATA_COPY = True
REFRESH_LOCAL_DATA = False
REQUIRE_GPU = True

# CC0 생활소음을 자동 수집합니다. 이미 충분히 있으면 다시 다운로드하지 않습니다.
PREPARE_OPEN_HARD_NEGATIVES = True
OPEN_HARD_NEGATIVES_PER_CATEGORY = 20
OPEN_HARD_NEGATIVE_LICENSE_MODE = "cc0_only"

# 논문 통계를 위한 반복 실행. 요구사항에 따라 5개 seed를 모두 사용합니다.
RUN_REPEATED_GROUP_EVALUATION = True
REPEATED_SPLIT_SEEDS = [42, 53, 64, 75, 86]

%pip -q install tensorflow-hub resampy soundfile pydub seaborn scikit-learn jinja2 requests
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import pickle
import platform
import random
import re
import shutil
import socket
import tarfile
import time
import unicodedata
import urllib.request
from urllib.parse import urlparse
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, ft2font
import numpy as np
import pandas as pd
import resampy
import requests
import seaborn as sns
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
from google.colab import drive, userdata
from IPython.display import display
from pydub import AudioSegment
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

SEED = 42
TARGET_SR = 16_000
YAMNET_FRAME_SECONDS = 0.96
MIN_SAMPLES = int(TARGET_SR * YAMNET_FRAME_SECONDS)
YAMNET_HANDLE = os.environ.get("HEARO_YAMNET_HANDLE", "https://tfhub.dev/google/yamnet/1")
UNKNOWN_LABEL = "비표적음"
TARGET_CATEGORIES = [
    "노크_목재",
    "노크_철재문",
    "도어락_개방음",
    "도어락_입력음",
    "사이렌_고저교번형",
    "사이렌_완만변조형",
    "사이렌_급속변조형",
    "아기 울음",
]
CATEGORIES = TARGET_CATEGORIES + [UNKNOWN_LABEL]
NUM_CLASSES = len(CATEGORIES)
UNKNOWN_ID = CATEGORIES.index(UNKNOWN_LABEL)
LEGACY_RELABEL = {
    "사이렌_삐뽀삐뽀": "사이렌_고저교번형",
    "사이렌_애애애애앵": "사이렌_완만변조형",
    "사이렌_철철철": "사이렌_급속변조형",
    "사이렌_안내음": UNKNOWN_LABEL,
}
SIREN_LABEL_DEFINITIONS = {
    "사이렌_고저교번형": {
        "international_term": "Hi-Lo",
        "operational_definition": "서로 다른 두 중심 주파수가 교대로 반복되는 사이렌",
    },
    "사이렌_완만변조형": {
        "international_term": "Wail",
        "operational_definition": "주파수가 비교적 완만하게 상승·하강하는 연속 사이렌",
    },
    "사이렌_급속변조형": {
        "international_term": "Yelp",
        "operational_definition": "주파수가 빠르게 상승·하강하며 반복되는 사이렌",
    },
}
KNOCK_CATEGORIES = ["노크_목재", "노크_철재문"]
KNOCK_IDS = np.asarray([CATEGORIES.index(label) for label in KNOCK_CATEGORIES], dtype=np.int64)
SIREN_CATEGORIES = [label for label in TARGET_CATEGORIES if label.startswith("사이렌_")]
SIREN_IDS = np.asarray([CATEGORIES.index(label) for label in SIREN_CATEGORIES], dtype=np.int64)
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a"}

PIPELINE_REVISION = "yamnet-colab-paper-v3.2-matched-seeds-r1"
OOF_CACHE_REVISION = "yamnet-colab-paper-v3.1-labels-r1"
PIPELINE_STARTED = time.perf_counter()
RUN_MODE = str(globals().get("RUN_MODE", os.environ.get("HEARO_RUN_MODE", "fast"))).strip().lower()
if RUN_MODE not in {"fast", "paper"}:
    raise ValueError("RUN_MODE은 'fast' 또는 'paper'여야 합니다.")
USE_LOCAL_DATA_COPY = bool(globals().get("USE_LOCAL_DATA_COPY", True))
REFRESH_LOCAL_DATA = bool(globals().get("REFRESH_LOCAL_DATA", False))
REQUIRE_GPU = bool(globals().get("REQUIRE_GPU", True))

DRIVE_DATA_DIR = Path(
    os.environ.get("HEARO_COLAB_DATA_DIR", "/content/drive/MyDrive/소리 정리")
)
LOCAL_DATA_DIR = Path("/content/hearo_data")
DATA_DIR = LOCAL_DATA_DIR if USE_LOCAL_DATA_COPY else DRIVE_DATA_DIR
METADATA_PATH = DATA_DIR / "metadata.csv"
OUTPUT_DIR = Path(
    os.environ.get(
        "HEARO_COLAB_OUTPUT_DIR",
        "/content/drive/MyDrive/Hearo_model_colab_paper_v3_2_newdata_20260925",
    )
)
MODEL_DIR = OUTPUT_DIR
CHECKPOINT_ROOT = OUTPUT_DIR / "checkpoints"
LOCAL_CACHE_ROOT = Path("/content/hearo_fast_cache")
CACHE_DIR = LOCAL_CACHE_ROOT / "embedding_cache"
WAVEFORM_CACHE_DIR = LOCAL_CACHE_ROOT / "waveform_cache"
# OOF 학습 로직은 v3와 동일하므로 검증된 v3 CV 캐시를 재사용합니다.
CV_CACHE_DIR = Path(
    "/content/drive/MyDrive/Hearo_model_colab_paper_v3/checkpoints/cv_family_cache"
)
FIG_DIR = OUTPUT_DIR / "figures"
PAPER_DIR = OUTPUT_DIR / "paper_metrics"
FONT_DIR = LOCAL_CACHE_ROOT / "fonts"
EMBEDDING_CACHE_ARCHIVE = CHECKPOINT_ROOT / "embedding_cache.tar"
LEGACY_V3_EMBEDDING_CACHE_ARCHIVE = Path(
    "/content/drive/MyDrive/Hearo_model_colab_paper_v3/checkpoints/embedding_cache.tar"
)
LEGACY_V22_EMBEDDING_CACHE_ARCHIVE = Path(
    "/content/drive/MyDrive/Hearo_model_colab_paper_v2_2/checkpoints/embedding_cache.tar"
)
LOCAL_COPY_MARKER = LOCAL_DATA_DIR / ".hearo_copy_complete"

if RUN_MODE == "fast":
    INNER_FOLDS = 3
    MAX_EPOCHS = 35
    BATCH_SIZE = 128
    PREDICT_BATCH_SIZE = 1_024
    PREDICT_CHUNK_FRAMES = 8_192
    EARLY_STOPPING_PATIENCE = 5
    LR_PATIENCE = 2
    AUGMENTATIONS_PER_FILE = 0
    BOOTSTRAP_ITERATIONS = 5_000
else:
    INNER_FOLDS = 4
    MAX_EPOCHS = 80
    BATCH_SIZE = 128
    PREDICT_BATCH_SIZE = 1_024
    PREDICT_CHUNK_FRAMES = 8_192
    EARLY_STOPPING_PATIENCE = 10
    LR_PATIENCE = 4
    AUGMENTATIONS_PER_FILE = 0
    BOOTSTRAP_ITERATIONS = 10_000

MAX_FALSE_ALERT_RATE = 0.02
MIN_PROMOTION = 0.005
HARD_NEGATIVE_MIN_CONFIDENCE = 0.25
SEMI_HARD_MIN_TARGET_PROBABILITY = 0.10
# 사전 등록된 최종 head 초기화. 모든 방법이 split별로 정확히 같은 seed를 공유합니다.
FINAL_TRAINING_SEED_OFFSETS = (1_000, 2_000, 3_000)


def final_training_seeds(outer_split_seed: int) -> tuple[int, ...]:
    seeds = tuple(int(outer_split_seed) + offset for offset in FINAL_TRAINING_SEED_OFFSETS)
    if len(seeds) != 3 or len(set(seeds)) != 3:
        raise AssertionError("최종 학습 초기화 seed는 서로 다른 3개여야 합니다.")
    return seeds

V3_ARM_SPECS = {
    "baseline": {
        "display_name": "기준선",
        "selection_policy": "none",
        "hard_negative_weight": 1.0,
    },
    "raw_top1_2x": {
        "display_name": "실제 raw top-1 오분류 2.0배",
        "selection_policy": "raw_top1_error",
        "hard_negative_weight": 2.0,
    },
    "raw_plus_semihard_p010_1p5x": {
        "display_name": "raw 오분류 + P(target)≥0.10, 1.5배",
        "selection_policy": "raw_plus_semihard_p010",
        "hard_negative_weight": 1.5,
    },
    "quota60_2x": {
        "display_name": "환경별 할당량 60개, 2.0배",
        "selection_policy": "environment_quota_60",
        "hard_negative_weight": 2.0,
    },
}
if tuple(V3_ARM_SPECS) != (
    "baseline", "raw_top1_2x", "raw_plus_semihard_p010_1p5x", "quota60_2x"
):
    raise AssertionError("v3 비교군 순서 또는 이름이 변경되었습니다.")
if [V3_ARM_SPECS[key]["hard_negative_weight"] for key in V3_ARM_SPECS] != [1.0, 2.0, 1.5, 2.0]:
    raise AssertionError("v3 비교군 가중치가 사전 정의와 다릅니다.")
MIN_HARD_NEGATIVES_TOTAL = 60
MIN_HARD_NEGATIVES_PER_ENV_CATEGORY = 8
MIN_OPEN_HARD_NEGATIVE_TEST_PER_CATEGORY = 2
ACTIVE_TRAINING_SEED = SEED

PREPARE_OPEN_HARD_NEGATIVES = bool(globals().get("PREPARE_OPEN_HARD_NEGATIVES", True))
OPEN_HARD_NEGATIVES_PER_CATEGORY = int(globals().get("OPEN_HARD_NEGATIVES_PER_CATEGORY", 20))
OPEN_HARD_NEGATIVE_LICENSE_MODE = str(globals().get("OPEN_HARD_NEGATIVE_LICENSE_MODE", "cc0_only"))
RUN_REPEATED_GROUP_EVALUATION = bool(globals().get("RUN_REPEATED_GROUP_EVALUATION", True))
REPEATED_SPLIT_SEEDS = list(globals().get("REPEATED_SPLIT_SEEDS", [42, 53, 64, 75, 86]))
if len(set(REPEATED_SPLIT_SEEDS)) < 5:
    raise ValueError("REPEATED_SPLIT_SEEDS에는 서로 다른 seed가 5개 이상 필요합니다.")
if BOOTSTRAP_ITERATIONS < 5_000:
    raise ValueError("BOOTSTRAP_ITERATIONS는 5,000회 이상이어야 합니다.")
if OPEN_HARD_NEGATIVE_LICENSE_MODE != "cc0_only":
    raise ValueError("이 버전은 저작권 위험을 낮추기 위해 cc0_only만 허용합니다.")

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

def _font_has_required_korean(font_path: Path) -> bool:
    try:
        face = ft2font.FT2Font(str(font_path))
        required = "".join(CATEGORIES) + (
            "한글생활소음오탐률정확도임계값클래스개발데이터학습배포모델"
            "원본그룹분리테스트혼동행렬예측실제표적완전보정"
        )
        return all(ord(char) in face.get_charmap() for char in required)
    except Exception:
        return False


def configure_korean_font() -> tuple[str, Path]:
    # 한글 글리프를 실제로 검증하고 없으면 Noto Sans KR을 내려받습니다.
    FONT_DIR.mkdir(parents=True, exist_ok=True)
    candidates: list[Path] = []
    configured = os.environ.get("KOREAN_FONT_PATH", "").strip()
    if configured:
        candidates.append(Path(configured).expanduser())

    search_roots = [
        Path("/usr/share/fonts"),
        Path("/usr/local/share/fonts"),
        Path.home() / ".local/share/fonts",
        Path.home() / "Library/Fonts",
        Path("/Library/Fonts"),
    ]
    preferred_fragments = ("notosanskr", "notosanscjk", "nanumgothic", "applegothic")
    discovered: list[Path] = []
    for root in search_roots:
        if root.exists():
            discovered.extend(root.rglob("*.ttf"))
            discovered.extend(root.rglob("*.otf"))
    candidates.extend(sorted(
        discovered,
        key=lambda path: (not any(part in path.name.lower() for part in preferred_fragments), str(path)),
    ))

    downloaded = FONT_DIR / "NotoSansKR.ttf"
    if downloaded.exists():
        candidates.insert(0, downloaded)

    selected = next((path for path in candidates if path.is_file() and _font_has_required_korean(path)), None)
    if selected is None:
        font_url = (
            "https://raw.githubusercontent.com/google/fonts/main/ofl/notosanskr/"
            "NotoSansKR%5Bwght%5D.ttf"
        )
        try:
            temporary = downloaded.with_suffix(".download")
            urllib.request.urlretrieve(font_url, temporary)
            if temporary.stat().st_size < 100_000 or not _font_has_required_korean(temporary):
                raise RuntimeError("다운로드한 폰트의 크기 또는 한글 글리프 검증에 실패했습니다.")
            temporary.replace(downloaded)
            selected = downloaded
        except Exception as exc:
            raise RuntimeError(
                "한글 폰트를 찾거나 내려받지 못했습니다. Noto Sans KR/NanumGothic TTF를 설치하거나 "
                "KOREAN_FONT_PATH를 지정하세요. 그림을 저장하기 전에 중단하여 글자 깨짐을 방지했습니다."
            ) from exc

    font_manager.fontManager.addfont(str(selected))
    font_name = font_manager.FontProperties(fname=str(selected)).get_name()
    resolved_path = Path(font_manager.findfont(font_name, fallback_to_default=False))
    if not _font_has_required_korean(resolved_path):
        resolved_path = selected
    if not _font_has_required_korean(resolved_path):
        raise RuntimeError(f"선택한 폰트에 필수 한글 글리프가 없습니다: {resolved_path}")

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": [font_name],
        "axes.unicode_minus": False,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    })
    print(f"한글 폰트 검증 완료: {font_name} ({resolved_path})")
    return font_name, resolved_path


GPU_DEVICES = tf.config.list_physical_devices("GPU")
if REQUIRE_GPU and not GPU_DEVICES:
    raise RuntimeError(
        "GPU가 감지되지 않았습니다. Colab 메뉴에서 런타임 > 런타임 유형 변경 > "
        "T4 GPU(또는 사용 가능한 GPU)를 선택하세요."
    )

drive.mount("/content/drive")

KOREAN_FONT_NAME, KOREAN_FONT_PATH = configure_korean_font()


def apply_korean_font(fig: mpl.figure.Figure) -> None:
    """제목·축·눈금·범례·heatmap 주석에 한글 폰트를 저장 직전에 강제 적용합니다."""
    for ax in fig.axes:
        text_items = [ax.title, ax.xaxis.label, ax.yaxis.label]
        text_items.extend(ax.get_xticklabels())
        text_items.extend(ax.get_yticklabels())
        text_items.extend(ax.texts)
        legend = ax.get_legend()
        if legend is not None:
            text_items.extend(legend.get_texts())
            text_items.append(legend.get_title())
        for text_item in text_items:
            text_item.set_fontfamily(KOREAN_FONT_NAME)


def save_report_figure(fig: mpl.figure.Figure, filename: str) -> Path:
    """한글 폰트를 다시 적용하고 보고서용 PNG가 정상 생성됐는지 검증합니다."""
    apply_korean_font(fig)
    fig.tight_layout()
    output_path = FIG_DIR / filename
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor="white")
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"그래프 저장 실패: {output_path}")
    print(f"그래프 저장 완료: {output_path}")
    return output_path

def _safe_extract_cache(archive_path: Path, destination: Path) -> int:
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()
    with tarfile.open(archive_path, "r") as archive:
        members = archive.getmembers()
        for member in members:
            target = (destination / member.name).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f"안전하지 않은 캐시 경로: {member.name}")
        archive.extractall(destination)
    return len(members)


def restore_embedding_cache() -> int:
    if any(CACHE_DIR.glob("*.npz")):
        return 0
    archive_path = next((path for path in [
        EMBEDDING_CACHE_ARCHIVE,
        LEGACY_V3_EMBEDDING_CACHE_ARCHIVE,
        LEGACY_V22_EMBEDDING_CACHE_ARCHIVE,
    ] if path.exists()), EMBEDDING_CACHE_ARCHIVE)
    if not archive_path.exists():
        return 0
    restored = _safe_extract_cache(archive_path, CACHE_DIR)
    source_name = (
        "v3.2" if archive_path == EMBEDDING_CACHE_ARCHIVE
        else "v3 재사용" if archive_path == LEGACY_V3_EMBEDDING_CACHE_ARCHIVE
        else "v2.2 재사용"
    )
    print(f"Drive embedding 캐시 복원({source_name}): {restored}개")
    return restored


def snapshot_embedding_cache() -> int:
    cache_files = sorted(CACHE_DIR.glob("*.npz"))
    if not cache_files:
        return 0
    temporary = EMBEDDING_CACHE_ARCHIVE.with_suffix(".tmp.tar")
    with tarfile.open(temporary, "w") as archive:
        for path in cache_files:
            archive.add(path, arcname=path.name, recursive=False)
    temporary.replace(EMBEDDING_CACHE_ARCHIVE)
    print(f"Drive embedding 캐시 저장: {len(cache_files)}개")
    return len(cache_files)


for directory in [
    OUTPUT_DIR, MODEL_DIR, CHECKPOINT_ROOT, LOCAL_CACHE_ROOT, CACHE_DIR,
    WAVEFORM_CACHE_DIR, CV_CACHE_DIR, FIG_DIR, PAPER_DIR, FONT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)



OPEN_HARD_NEGATIVE_DIR = DRIVE_DATA_DIR / "생활소음_오픈데이터_CC0"
OPEN_HARD_NEGATIVE_MANIFEST = DRIVE_DATA_DIR / "open_hard_negative_sources.csv"
OPEN_HARD_NEGATIVE_ATTRIBUTION = DRIVE_DATA_DIR / "OPEN_DATA_PROVENANCE.md"
OPEN_HARD_NEGATIVE_QUERIES = {
    "발걸음": ["indoor footsteps", "walking footsteps"],
    "문_여닫기": ["door opening closing", "door slam", "sliding door"],
    "열쇠": ["keys jingling", "keychain jingle"],
    "식기": ["dishes cutlery", "plates utensils", "pots pans kitchen"],
    # 저작권이 있는 실제 방송 녹음 대신 일반 대화·웅성거림으로 TV/음성 환경을 모사합니다.
    "TV_음성": [
        "conversation", "speech babble", "background speech", "people talking",
        "muffled voices", "distant voices", "room chatter", "office chatter",
        "restaurant chatter", "crowd voices", "murmur", "unintelligible speech",
    ],
    "전자제품_알림음": ["appliance beep", "microwave beep", "electronic chime"],
}
OPEN_DATA_COLUMNS = [
    "relative_path", "label", "group_id", "hard_negative_candidate",
    "environment_category", "source_dataset", "source_id", "source_url",
    "source_username", "license", "attribution", "download_url",
]
BLOCKED_RIGHTS_TERMS = {
    "movie", "film", "youtube", "netflix", "television show", "tv show",
    "song", "music", "game audio", "commercial", "advertisement",
}


def _colab_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""


def _safe_source_text(value: Any) -> str:
    return re.sub(r"[\r\n\t]+", " ", str(value or "")).strip()


def _is_rights_safe_candidate(item: dict[str, Any]) -> bool:
    license_name = _safe_source_text(item.get("license")).lower()
    if "creative commons 0" not in license_name and "publicdomain/zero" not in license_name:
        return False
    preference = _safe_source_text(item.get("gen_ai_preference")).lower()
    if preference and preference != "no-additional-preferences":
        return False
    searchable = " ".join([
        _safe_source_text(item.get("name")),
        _safe_source_text(item.get("description")),
        " ".join(map(str, item.get("tags") or [])),
    ]).lower()
    return not any(term in searchable for term in BLOCKED_RIGHTS_TERMS)


def _write_open_data_provenance(frame: pd.DataFrame) -> None:
    lines = [
        "# Hearo 공개 생활소음 출처표",
        "",
        "- 수집 경로: Freesound API v2",
        "- 허용 라이선스: Creative Commons 0 (CC0)만",
        "- 용도: 비표적 생활소음 및 development OOF 하드/준하드 네거티브 후보",
        "- 주의: 최종 논문 제출 전 파일 내용과 원 출처 페이지를 사람이 다시 확인할 것",
        "",
        "|분류|원본 ID|업로더|라이선스|원본 URL|",
        "|---|---:|---|---|---|",
    ]
    for row in frame.sort_values(["environment_category", "source_id"]).itertuples(index=False):
        lines.append(
            f"|{row.environment_category}|{row.source_id}|{row.source_username}|"
            f"{row.license}|{row.source_url}|"
        )
    OPEN_HARD_NEGATIVE_ATTRIBUTION.write_text("\n".join(lines) + "\n", encoding="utf-8")


def _sync_open_data_into_metadata(frame: pd.DataFrame) -> None:
    metadata_path = DRIVE_DATA_DIR / "metadata.csv"
    existing = pd.read_csv(metadata_path, dtype=str, keep_default_na=False)
    additions = frame[[
        "relative_path", "label", "group_id", "hard_negative_candidate",
        "environment_category", "source_dataset", "source_id", "source_url",
        "source_username", "license", "attribution",
    ]].copy()
    all_columns = list(existing.columns)
    for column in additions.columns:
        if column not in all_columns:
            all_columns.append(column)
    for column in all_columns:
        if column not in existing:
            existing[column] = ""
        if column not in additions:
            additions[column] = ""
    existing = existing[~existing["relative_path"].isin(additions["relative_path"])]
    combined = pd.concat([existing[all_columns], additions[all_columns]], ignore_index=True)
    combined.to_csv(metadata_path, index=False, encoding="utf-8-sig")


def prepare_open_hard_negative_audio() -> pd.DataFrame:
    """CC0 생활소음을 수집하고 metadata.csv에 비표적 후보로 등록합니다."""
    OPEN_HARD_NEGATIVE_DIR.mkdir(parents=True, exist_ok=True)
    if OPEN_HARD_NEGATIVE_MANIFEST.exists():
        manifest = pd.read_csv(OPEN_HARD_NEGATIVE_MANIFEST, dtype=str, keep_default_na=False)
    else:
        manifest = pd.DataFrame(columns=OPEN_DATA_COLUMNS)

    def category_count(category: str) -> int:
        if manifest.empty:
            return 0
        exists = manifest["relative_path"].map(lambda value: (DRIVE_DATA_DIR / value).is_file())
        return int(((manifest["environment_category"] == category) & exists).sum())

    missing_categories = [
        category for category in OPEN_HARD_NEGATIVE_QUERIES
        if category_count(category) < OPEN_HARD_NEGATIVES_PER_CATEGORY
    ]
    if missing_categories:
        token = _colab_secret("FREESOUND_API_KEY")
        if not token:
            raise RuntimeError(
                "Colab 왼쪽의 열쇠(비밀) 메뉴에 FREESOUND_API_KEY를 등록하세요. "
                "키는 https://freesound.org/apiv2/apply/ 에서 발급받습니다."
            )
        session = requests.Session()
        session.headers.update({"Authorization": f"Token {token}"})
        used_ids = set(manifest.get("source_id", pd.Series(dtype=str)).astype(str))

        for category, queries in OPEN_HARD_NEGATIVE_QUERIES.items():
            needed = OPEN_HARD_NEGATIVES_PER_CATEGORY - category_count(category)
            if needed <= 0:
                continue
            per_uploader: dict[str, int] = {}
            if not manifest.empty:
                current = manifest[manifest["environment_category"] == category]
                per_uploader.update(current["source_username"].value_counts().to_dict())

            for query in queries:
                for page in range(1, 6):
                    if needed <= 0:
                        break
                    response = session.get(
                        "https://freesound.org/apiv2/search/",
                        params={
                            "query": query,
                            "filter": 'license:"Creative Commons 0" duration:[0.8 TO 15]',
                            "sort": "rating_desc",
                            "page": page,
                            "page_size": 100,
                            "fields": (
                                "id,name,description,tags,username,license,duration,url,previews,"
                                "gen_ai_preference"
                            ),
                        },
                        timeout=60,
                    )
                    # Freesound는 범위를 벗어난 페이지에 404를 반환할 수 있습니다.
                    # 첫 페이지의 404는 실제 오류로 처리하되, 후속 페이지의 404는
                    # 검색 결과가 끝난 것으로 간주합니다.
                    if response.status_code == 404 and page > 1:
                        break
                    response.raise_for_status()
                    payload = response.json()
                    results = payload.get("results", [])
                    if not results:
                        break
                    for item in results:
                        source_id = str(item.get("id", ""))
                        username = _safe_source_text(item.get("username"))
                        if not source_id or source_id in used_ids or per_uploader.get(username, 0) >= 3:
                            continue
                        if not _is_rights_safe_candidate(item):
                            continue
                        preview_url = (item.get("previews") or {}).get("preview-hq-mp3")
                        if not preview_url:
                            continue
                        category_dir = OPEN_HARD_NEGATIVE_DIR / category
                        category_dir.mkdir(parents=True, exist_ok=True)
                        destination = category_dir / f"freesound_{source_id}.mp3"
                        audio_response = session.get(preview_url, timeout=120)
                        audio_response.raise_for_status()
                        if len(audio_response.content) < 2_000:
                            continue
                        destination.write_bytes(audio_response.content)
                        relative_path = destination.relative_to(DRIVE_DATA_DIR).as_posix()
                        username_hash = hashlib.sha256(username.encode("utf-8")).hexdigest()[:12]
                        row = {
                            "relative_path": relative_path,
                            "label": UNKNOWN_LABEL,
                            "group_id": f"freesound_uploader_{username_hash}",
                            "hard_negative_candidate": "true",
                            "environment_category": category,
                            "source_dataset": "Freesound_API_v2",
                            "source_id": source_id,
                            "source_url": _safe_source_text(item.get("url")),
                            "source_username": username,
                            "license": "CC0-1.0",
                            "attribution": f"Freesound {source_id}, uploaded by {username}, CC0-1.0",
                            "download_url": preview_url,
                        }
                        manifest = pd.concat([manifest, pd.DataFrame([row])], ignore_index=True)
                        manifest.to_csv(
                            OPEN_HARD_NEGATIVE_MANIFEST, index=False, encoding="utf-8-sig"
                        )
                        used_ids.add(source_id)
                        per_uploader[username] = per_uploader.get(username, 0) + 1
                        needed -= 1
                        if needed <= 0:
                            break

                    # 공식 API의 next가 null이면 다음 페이지를 요청하지 않습니다.
                    if not payload.get("next"):
                        break

            if needed > 0:
                raise RuntimeError(
                    f"CC0 생활소음 '{category}'가 {needed}개 부족합니다. "
                    "쿼리를 보완하거나 직접 녹음한 CC0 음원을 manifest에 추가하세요."
                )

    # 파일, 라이선스, 개수의 방어적 검증
    if manifest.empty or set(manifest["license"]) != {"CC0-1.0"}:
        raise RuntimeError("공개 생활소음 manifest가 비었거나 CC0 이외 라이선스가 포함됐습니다.")
    missing_files = [
        value for value in manifest["relative_path"] if not (DRIVE_DATA_DIR / value).is_file()
    ]
    if missing_files:
        raise FileNotFoundError(f"공개 생활소음 파일 누락: {missing_files[:10]}")
    counts = manifest.groupby("environment_category").size().reindex(
        OPEN_HARD_NEGATIVE_QUERIES, fill_value=0
    )
    if (counts < OPEN_HARD_NEGATIVES_PER_CATEGORY).any():
        raise RuntimeError(f"생활소음 카테고리별 최소 개수를 충족하지 못했습니다.\n{counts}")
    _sync_open_data_into_metadata(manifest)
    _write_open_data_provenance(manifest)
    print("CC0 생활소음 준비 완료:")
    display(counts.rename("files"))
    return manifest


if PREPARE_OPEN_HARD_NEGATIVES:
    open_hard_negative_sources = prepare_open_hard_negative_audio()
else:
    if not OPEN_HARD_NEGATIVE_MANIFEST.exists():
        raise FileNotFoundError(
            "PREPARE_OPEN_HARD_NEGATIVES=False이면 open_hard_negative_sources.csv가 미리 필요합니다."
        )
    open_hard_negative_sources = pd.read_csv(
        OPEN_HARD_NEGATIVE_MANIFEST, dtype=str, keep_default_na=False
    )
    _sync_open_data_into_metadata(open_hard_negative_sources)


if not (DRIVE_DATA_DIR / "metadata.csv").exists():
    raise FileNotFoundError(
        f"{DRIVE_DATA_DIR / 'metadata.csv'}가 없습니다. Google Drive 데이터 위치를 확인하세요."
    )

source_metadata_sha256 = hashlib.sha256(
    (DRIVE_DATA_DIR / "metadata.csv").read_bytes()
).hexdigest()
expected_copy_marker = [
    f"source={DRIVE_DATA_DIR}",
    f"metadata_sha256={source_metadata_sha256}",
]
local_copy_ready = False
if LOCAL_COPY_MARKER.exists():
    try:
        marker_lines = LOCAL_COPY_MARKER.read_text(encoding="utf-8").splitlines()
        local_copy_ready = marker_lines[:2] == expected_copy_marker
    except (OSError, UnicodeError, IndexError):
        local_copy_ready = False

if USE_LOCAL_DATA_COPY and (REFRESH_LOCAL_DATA or not local_copy_ready):
    copy_started = time.perf_counter()
    # metadata 변경 또는 중단된 복사를 감지하면 부분 복사본을 제거하고 다시 복사합니다.
    if LOCAL_DATA_DIR.exists():
        shutil.rmtree(LOCAL_DATA_DIR)
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)
    LOCAL_COPY_MARKER.write_text(
        "\n".join(expected_copy_marker)
        + f"\ncompleted_at={datetime.now(timezone.utc).isoformat()}\n",
        encoding="utf-8",
    )
    print(f"Drive -> 로컬 SSD 복사 완료: {time.perf_counter() - copy_started:.1f}초")

restore_embedding_cache()

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"{METADATA_PATH}가 없습니다.")
if any(ord(character) == 0xFFFD for character in str(DATA_DIR) + str(OUTPUT_DIR)):
    raise UnicodeError("입출력 경로에 Unicode replacement character(U+FFFD)가 있습니다.")

print("Pipeline revision:", PIPELINE_REVISION)
print("실행 모드:", RUN_MODE)
print("TensorFlow:", tf.__version__)
print("GPU:", GPU_DEVICES)
print("입력 경로:", DATA_DIR)
print("출력 경로:", OUTPUT_DIR)
print("로컬 캐시:", LOCAL_CACHE_ROOT)



## 2. 오디오 로드와 metadata.csv 검증

metadata.csv 필수 열은 relative_path, label, group_id입니다. relative_path는 DATA_DIR 기준 경로이며,
같은 녹음 세션 또는 같은 원본에서 파생된 파일은 반드시 같은 group_id를 사용해야 합니다.



In [ ]:
def nfc(value: Any) -> str:
    return unicodedata.normalize("NFC", str(value).strip())


def load_audio(path: str | Path, pad_short: bool = True) -> np.ndarray:
    """지원 음원을 mono 16 kHz float32 [-1, 1]로 변환합니다."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext not in AUDIO_EXTENSIONS:
        raise ValueError(f"지원하지 않는 확장자: {path}")

    if ext in {".mp3", ".m4a"}:
        if shutil.which("ffmpeg") is None and shutil.which("avconv") is None:
            raise RuntimeError("MP3/M4A 디코딩에 ffmpeg가 필요합니다. 위 설치 셀을 다시 실행해 ffmpeg를 설치하세요.")
        segment = AudioSegment.from_file(path)
        segment = segment.set_channels(1).set_frame_rate(TARGET_SR)
        raw = np.asarray(segment.get_array_of_samples())
        scale = float(1 << (8 * segment.sample_width - 1))
        waveform = raw.astype(np.float32) / scale
    else:
        waveform, sample_rate = sf.read(path, dtype="float32", always_2d=False)
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)
        if int(sample_rate) != TARGET_SR:
            waveform = resampy.resample(waveform, int(sample_rate), TARGET_SR)

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)
    if waveform.size == 0 or not np.all(np.isfinite(waveform)):
        raise ValueError(f"비어 있거나 유효하지 않은 음원: {path}")
    waveform = np.clip(waveform, -1.0, 1.0)
    if pad_short and len(waveform) < MIN_SAMPLES:
        waveform = np.pad(waveform, (0, MIN_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)


def waveform_cache_path(audio_hash: str) -> Path:
    return WAVEFORM_CACHE_DIR / f"{audio_hash}.npy"


def save_waveform_cache(audio_hash: str, waveform: np.ndarray) -> Path:
    path = waveform_cache_path(audio_hash)
    if path.exists():
        return path
    temporary = path.with_suffix(".tmp.npy")
    with temporary.open("wb") as file:
        np.save(file, np.asarray(waveform, dtype=np.float32), allow_pickle=False)
    os.replace(temporary, path)
    return path


def load_row_waveform(row: Any) -> np.ndarray:
    path = waveform_cache_path(str(row.audio_hash))
    if path.exists():
        return np.load(path, allow_pickle=False).astype(np.float32, copy=False)
    waveform = load_audio(row.filepath)
    if decoded_audio_hash(waveform) != str(row.audio_hash):
        raise RuntimeError(f"검증 이후 음원 내용이 변경되었습니다: {row.filepath}")
    save_waveform_cache(str(row.audio_hash), waveform)
    return waveform


def decoded_audio_hash(waveform: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(waveform, np.float32).tobytes()).hexdigest()


def build_normalized_file_index(root: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rel = nfc(path.relative_to(root).as_posix())
            if rel in index:
                raise ValueError(f"NFC 정규화 후 경로가 충돌합니다: {rel}")
            index[rel] = path
    return index


def validate_and_load_metadata(metadata_path: Path, data_dir: Path) -> pd.DataFrame:
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"{metadata_path}가 없습니다. relative_path,label,group_id 열을 만들어 주세요."
        )
    raw = pd.read_csv(metadata_path, dtype=str, keep_default_na=False)
    required = {"relative_path", "label", "group_id"}
    missing_columns = required - set(raw.columns)
    if missing_columns:
        raise ValueError(f"metadata.csv 필수 열 누락: {sorted(missing_columns)}")

    optional_defaults = {
        "hard_negative_candidate": "false",
        "environment_category": "",
        "source_dataset": "",
        "source_id": "",
        "source_url": "",
        "source_username": "",
        "license": "",
        "attribution": "",
    }
    for column, default in optional_defaults.items():
        if column not in raw.columns:
            raw[column] = default
    selected_columns = ["relative_path", "label", "group_id", *optional_defaults]
    df = raw[selected_columns].copy()
    df["hard_negative_candidate"] = (
        df["hard_negative_candidate"].astype(str).str.strip().str.lower()
        .isin({"1", "true", "yes", "y"})
    )
    for column in required:
        df[column] = df[column].map(nfc)
    if (df[list(required)] == "").any().any():
        raise ValueError("metadata.csv에 빈 relative_path, label 또는 group_id가 있습니다.")
    df["source_label"] = df["label"]
    df["label"] = df["label"].replace(LEGACY_RELABEL)
    relabel_count = int((df["source_label"] != df["label"]).sum())
    print(f"기존 메타데이터 라벨 {relabel_count}개를 논문용 표준 라벨로 재지정합니다.")
    if df["relative_path"].duplicated().any():
        duplicates = df.loc[df["relative_path"].duplicated(False), "relative_path"].tolist()
        raise ValueError(f"metadata.csv 중복 경로: {duplicates[:10]}")

    unexpected_labels = sorted(set(df["label"]) - set(CATEGORIES))
    missing_labels = sorted(set(CATEGORIES) - set(df["label"]))
    if unexpected_labels or missing_labels:
        raise ValueError(
            f"레이블 불일치. 예상 밖={unexpected_labels}, 데이터에 없음={missing_labels}"
        )

    file_index = build_normalized_file_index(data_dir)
    listed = set(df["relative_path"])
    missing_files = sorted(listed - set(file_index))
    unlisted_files = sorted(set(file_index) - listed)
    if missing_files or unlisted_files:
        raise ValueError(
            "metadata.csv와 실제 파일이 일치하지 않습니다. "
            f"없는 파일={missing_files[:10]}, 미등록 파일={unlisted_files[:10]}"
        )

    records = []
    hash_to_label: dict[str, str] = {}
    errors = []
    for row in df.itertuples(index=False):
        path = file_index[row.relative_path]
        try:
            waveform = load_audio(path)
            audio_hash = decoded_audio_hash(waveform)
            save_waveform_cache(audio_hash, waveform)
            previous_label = hash_to_label.get(audio_hash)
            if previous_label is not None and previous_label != row.label:
                raise ValueError(f"동일 음원이 서로 다른 레이블에 존재: {previous_label}, {row.label}")
            hash_to_label[audio_hash] = row.label
            records.append({
                "relative_path": row.relative_path,
                "filepath": str(path),
                "label": row.label,
                "source_label": row.source_label,
                "label_id": CATEGORIES.index(row.label),
                "group_id": row.group_id,
                "audio_hash": audio_hash,
                "duration_seconds": len(waveform) / TARGET_SR,
                "rms": float(np.sqrt(np.mean(np.square(waveform)) + 1e-12)),
                "peak": float(np.max(np.abs(waveform))),
                "is_hard_negative": False,
                "hard_negative_candidate": bool(row.hard_negative_candidate),
                "environment_category": row.environment_category,
                "source_dataset": row.source_dataset,
                "source_id": row.source_id,
                "source_url": row.source_url,
                "source_username": row.source_username,
                "license": row.license,
                "attribution": row.attribution,
            })
        except Exception as exc:
            errors.append(f"{row.relative_path}: {exc}")
    if errors:
        raise ValueError("음원 검증 실패:\n" + "\n".join(errors[:30]))

    validated = pd.DataFrame(records).sort_values("relative_path").reset_index(drop=True)
    duplicate_hashes = validated[validated["audio_hash"].duplicated(False)]
    if not duplicate_hashes.empty:
        print("경고: 동일 레이블 내 완전 중복 음원이 있습니다.")
        display(duplicate_hashes[["relative_path", "label", "group_id", "audio_hash"]])
    return validated


metadata = validate_and_load_metadata(METADATA_PATH, DATA_DIR)
display(metadata.groupby("label").agg(files=("relative_path", "size"), groups=("group_id", "nunique"), seconds=("duration_seconds", "sum")))
metadata.to_csv(OUTPUT_DIR / "validated_manifest.csv", index=False, encoding="utf-8-sig")



## 3. 누수 없는 outer test / inner CV 분할



In [ ]:
REQUIRED_CLASS_IDS = set(range(NUM_CLASSES))


def missing_classes(df: pd.DataFrame, indices: np.ndarray) -> set[int]:
    return REQUIRED_CLASS_IDS - set(df.iloc[indices]["label_id"].astype(int))


def groups_per_class(df: pd.DataFrame, indices: np.ndarray | None = None) -> pd.Series:
    subset = df if indices is None else df.iloc[indices]
    return (
        subset.groupby("label_id")["group_id"].nunique()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .astype(int)
    )


def feasible_splits(df: pd.DataFrame, requested: int, name: str) -> int:
    counts = groups_per_class(df)
    count = int(min(requested, counts.min()))
    if count < 3:
        readable = pd.Series(counts.to_numpy(), index=CATEGORIES)
        raise ValueError(
            f"{name} grouped split에는 클래스마다 최소 3개 group_id가 필요합니다.\n{readable}"
        )
    return count


def distribution_distance(labels: np.ndarray, indices: np.ndarray) -> float:
    full = np.bincount(labels, minlength=NUM_CLASSES) / len(labels)
    part = np.bincount(labels[indices], minlength=NUM_CLASSES) / len(indices)
    return float(np.abs(full - part).sum())


def holdout_score(df: pd.DataFrame, test_idx: np.ndarray, test_size: float) -> float:
    labels = df["label_id"].to_numpy(dtype=np.int64)
    full_counts = np.bincount(labels, minlength=NUM_CLASSES)
    test_counts = np.bincount(labels[test_idx], minlength=NUM_CLASSES)
    per_class_fraction = test_counts / np.maximum(full_counts, 1)
    class_ratio_error = float(np.mean(np.abs(per_class_fraction - test_size)))
    row_ratio_error = abs(len(test_idx) / len(df) - test_size)
    group_ratio = df.iloc[test_idx]["group_id"].nunique() / df["group_id"].nunique()
    group_ratio_error = abs(group_ratio - test_size)
    return class_ratio_error + 0.25 * row_ratio_error + 0.05 * group_ratio_error


def open_noise_category_counts(df: pd.DataFrame, indices: np.ndarray) -> pd.Series:
    subset = df.iloc[indices]
    subset = subset[subset["hard_negative_candidate"].astype(bool)]
    return subset.groupby("environment_category").size().reindex(
        OPEN_HARD_NEGATIVE_QUERIES, fill_value=0
    ).astype(int)


def select_outer_holdout(
    df: pd.DataFrame, test_size: float = 0.20, trials: int = 512, seed: int = SEED
) -> tuple[np.ndarray, np.ndarray, float]:
    all_group_counts = groups_per_class(df)
    if (all_group_counts < 4).any():
        readable = pd.Series(all_group_counts.to_numpy(), index=CATEGORIES)
        raise ValueError(
            "최종 test와 최소 3-fold CV를 함께 만들려면 클래스마다 최소 4개 group_id가 필요합니다."
            f"\n{readable}"
        )

    splitter = GroupShuffleSplit(n_splits=trials, test_size=test_size, random_state=seed)
    best: tuple[float, np.ndarray, np.ndarray] | None = None
    for train_idx, test_idx in splitter.split(df, df["label_id"], df["group_id"]):
        if missing_classes(df, train_idx) or missing_classes(df, test_idx):
            continue
        development_open_counts = open_noise_category_counts(df, train_idx)
        test_open_counts = open_noise_category_counts(df, test_idx)
        if (development_open_counts < MIN_HARD_NEGATIVES_PER_ENV_CATEGORY).any():
            continue
        if (test_open_counts < MIN_OPEN_HARD_NEGATIVE_TEST_PER_CATEGORY).any():
            continue
        # test 격리 뒤 development에 클래스별 group_id를 3개 이상 남겨야 합니다.
        if (groups_per_class(df, train_idx) < 3).any():
            continue
        score = holdout_score(df, test_idx, test_size)
        if best is None or score < best[0] - 1e-12:
            best = (score, train_idx.copy(), test_idx.copy())

    if best is None:
        raise ValueError(
            "모든 클래스를 포함하면서 development에 클래스별 group_id 3개를 남기는 "
            "outer test split을 찾지 못했습니다. 희소 클래스 또는 생활소음 범주의 독립 uploader/group을 추가하세요."
        )
    return best[1], best[2], best[0]


def select_inner_splits(
    df: pd.DataFrame, n_splits: int, trials: int = 64, seed: int = SEED + 1
) -> tuple[list[tuple[np.ndarray, np.ndarray]], float, int]:
    labels = df["label_id"].to_numpy(dtype=np.int64)
    best: tuple[float, list[tuple[np.ndarray, np.ndarray]], int] | None = None
    for offset in range(trials):
        candidate_seed = seed + offset
        splitter = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=candidate_seed
        )
        candidate = list(splitter.split(df, labels, df["group_id"]))
        if any(
            missing_classes(df, train_idx) or missing_classes(df, val_idx)
            for train_idx, val_idx in candidate
        ):
            continue
        fold_sizes = np.asarray([len(val_idx) for _, val_idx in candidate], dtype=np.float64)
        score = float(np.mean([distribution_distance(labels, val_idx) for _, val_idx in candidate]))
        score += float(np.std(fold_sizes) / len(df))
        if best is None or score < best[0] - 1e-12:
            best = (score, [(a.copy(), b.copy()) for a, b in candidate], candidate_seed)

    if best is None:
        raise ValueError(
            f"모든 클래스가 train/validation에 존재하는 {n_splits}-fold grouped CV를 "
            "찾지 못했습니다. 희소 클래스의 독립 원본을 추가하세요."
        )
    return best[1], best[0], best[2]


outer_train_idx, test_idx, outer_score = select_outer_holdout(metadata)
# 같은 seed에서 선택 결과가 완전히 재현되는지 즉시 확인합니다.
outer_train_repeat, test_repeat, outer_score_repeat = select_outer_holdout(metadata)
assert np.array_equal(outer_train_idx, outer_train_repeat)
assert np.array_equal(test_idx, test_repeat)
assert np.isclose(outer_score, outer_score_repeat)

dev_df = metadata.iloc[outer_train_idx].reset_index(drop=True)
test_df = metadata.iloc[test_idx].reset_index(drop=True)
assert set(dev_df["group_id"]).isdisjoint(set(test_df["group_id"]))

for split_name, split_df in [("development", dev_df), ("test", test_df)]:
    missing = REQUIRED_CLASS_IDS - set(split_df["label_id"].astype(int))
    if missing:
        raise AssertionError(f"{split_name} split에 빠진 클래스: {[CATEGORIES[i] for i in sorted(missing)]}")

inner_splits_count = feasible_splits(dev_df, INNER_FOLDS, "inner")
inner_splits, inner_score, inner_seed = select_inner_splits(dev_df, inner_splits_count)
inner_repeat, inner_score_repeat, inner_seed_repeat = select_inner_splits(dev_df, inner_splits_count)
assert inner_seed == inner_seed_repeat and np.isclose(inner_score, inner_score_repeat)
assert all(
    np.array_equal(a_train, b_train) and np.array_equal(a_val, b_val)
    for (a_train, a_val), (b_train, b_val) in zip(inner_splits, inner_repeat)
)
for train_idx, val_idx in inner_splits:
    assert set(dev_df.iloc[train_idx]["group_id"]).isdisjoint(set(dev_df.iloc[val_idx]["group_id"]))
    assert not missing_classes(dev_df, train_idx)
    assert not missing_classes(dev_df, val_idx)

split_manifest = metadata[["relative_path", "label", "group_id"]].copy()
split_manifest["split"] = np.where(split_manifest["relative_path"].isin(test_df["relative_path"]), "test", "development")
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False, encoding="utf-8-sig")
print(
    f"Development={len(dev_df)}, Test={len(test_df)} "
    f"({len(test_df) / len(metadata):.1%}), inner folds={inner_splits_count}, inner seed={inner_seed}"
)
display(pd.DataFrame({
    "development_files": dev_df.groupby("label").size().reindex(CATEGORIES),
    "test_files": test_df.groupby("label").size().reindex(CATEGORIES),
    "development_groups": dev_df.groupby("label")["group_id"].nunique().reindex(CATEGORIES),
    "test_groups": test_df.groupby("label")["group_id"].nunique().reindex(CATEGORIES),
}))



## 4. YAMNet 프레임 특징과 안전한 증강 캐시



In [ ]:
print("YAMNet 로딩 중...")
yamnet_model = hub.load(YAMNET_HANDLE)
print("YAMNet 로드 완료")


def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def zero_filled_shift(waveform: np.ndarray, shift_samples: int) -> np.ndarray:
    shifted = np.zeros_like(waveform)
    if shift_samples > 0:
        shifted[shift_samples:] = waveform[:-shift_samples]
    elif shift_samples < 0:
        shifted[:shift_samples] = waveform[-shift_samples:]
    else:
        shifted[:] = waveform
    return shifted


def fit_noise(noise: np.ndarray, length: int, rng: np.random.Generator) -> np.ndarray:
    if len(noise) < length:
        repeats = int(np.ceil(length / len(noise)))
        noise = np.tile(noise, repeats)
    start = int(rng.integers(0, max(1, len(noise) - length + 1)))
    return noise[start:start + length]


def augment_waveform(
    waveform: np.ndarray,
    seed: int,
    noise_paths: list[str],
    allow_noise_mix: bool,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    augmented = waveform.astype(np.float32).copy()

    gain_db = float(rng.uniform(-6.0, 6.0))
    augmented *= np.float32(10.0 ** (gain_db / 20.0))

    max_shift = int(0.1 * TARGET_SR)
    shift = int(rng.integers(-max_shift, max_shift + 1))
    augmented = zero_filled_shift(augmented, shift)

    speed = float(rng.uniform(0.95, 1.05))
    stretched_sr = max(1, int(round(TARGET_SR / speed)))
    augmented = resampy.resample(augmented, TARGET_SR, stretched_sr).astype(np.float32)

    if allow_noise_mix and noise_paths and rng.random() < 0.75:
        noise_path = noise_paths[int(rng.integers(0, len(noise_paths)))]
        noise = fit_noise(load_audio(noise_path), len(augmented), rng)
        signal_rms = float(np.sqrt(np.mean(augmented ** 2) + 1e-12))
        noise_rms = float(np.sqrt(np.mean(noise ** 2) + 1e-12))
        snr_db = float(rng.uniform(10.0, 30.0))
        noise_gain = signal_rms / (max(noise_rms, 1e-8) * 10.0 ** (snr_db / 20.0))
        augmented = augmented + noise * np.float32(noise_gain)

    if len(augmented) < MIN_SAMPLES:
        augmented = np.pad(augmented, (0, MIN_SAMPLES - len(augmented)))
    return np.clip(augmented, -1.0, 1.0).astype(np.float32)


@tf.function(
    input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)],
    reduce_retracing=True,
)
def yamnet_features_graph(waveform: tf.Tensor) -> tf.Tensor:
    _, embeddings, _ = yamnet_model(waveform)
    return embeddings


def yamnet_features(waveform: np.ndarray) -> np.ndarray:
    embeddings = yamnet_features_graph(tf.convert_to_tensor(waveform, tf.float32))
    return embeddings.numpy().astype(np.float32)


CACHE_WRITE_COUNT = 0


def cached_features(
    row: Any,
    augmentation_index: int = 0,
    noise_paths: list[str] | None = None,
) -> np.ndarray:
    global CACHE_WRITE_COUNT
    profile = "clean" if augmentation_index == 0 else f"light_{augmentation_index}"

    # clean cache는 metadata 검증에서 계산한 decoded hash로 먼저 조회합니다.
    if augmentation_index == 0:
        waveform_hash = str(row.audio_hash)
        cache_key = hashlib.sha256(
            f"{YAMNET_HANDLE}|{waveform_hash}|{profile}".encode("utf-8")
        ).hexdigest()
        cache_path = CACHE_DIR / f"{cache_key}.npz"
        if cache_path.exists():
            try:
                with np.load(cache_path, allow_pickle=False) as cached:
                    return cached["embeddings"]
            except Exception:
                cache_path.unlink(missing_ok=True)
        waveform = load_row_waveform(row)
    else:
        waveform = load_row_waveform(row)
        waveform = augment_waveform(
            waveform,
            stable_seed(SEED, row.audio_hash, augmentation_index),
            noise_paths or [],
            allow_noise_mix=(row.label != UNKNOWN_LABEL),
        )
        waveform_hash = decoded_audio_hash(waveform)
        cache_key = hashlib.sha256(
            f"{YAMNET_HANDLE}|{waveform_hash}|{profile}".encode("utf-8")
        ).hexdigest()
        cache_path = CACHE_DIR / f"{cache_key}.npz"
        if cache_path.exists():
            try:
                with np.load(cache_path, allow_pickle=False) as cached:
                    return cached["embeddings"]
            except Exception:
                cache_path.unlink(missing_ok=True)

    embeddings = yamnet_features(waveform)
    temporary_path = cache_path.with_suffix(".tmp.npz")
    # Colab 로컬 SSD에서는 압축보다 읽기/쓰기 속도를 우선합니다.
    np.savez(temporary_path, embeddings=embeddings)
    os.replace(temporary_path, cache_path)
    CACHE_WRITE_COUNT += 1
    return embeddings


# clean 특징은 split과 무관한 frozen YAMNet 변환이므로 한 번만 캐시합니다.
embedding_started = time.perf_counter()
for index, row in enumerate(metadata.itertuples(index=False), start=1):
    cached_features(row)
    if index % 25 == 0 or index == len(metadata):
        print(f"Clean embedding cache: {index}/{len(metadata)}")

if CACHE_WRITE_COUNT > 0 or not EMBEDDING_CACHE_ARCHIVE.exists():
    snapshot_embedding_cache()
print(f"Clean embedding 단계: {time.perf_counter() - embedding_started:.1f}초")



## 5. 분류기, 프레임 집계 및 운영 지표



In [ ]:
@dataclass(frozen=True)
class FamilyConfig:
    name: str
    head: str
    representation: str
    augmentation: str = "none"
    class_weighting: bool = False
    hard_negative_weight: float = 1.0


def balanced_class_weights(rows: pd.DataFrame) -> dict[int, float]:
    counts = rows.groupby("label_id").size().reindex(range(NUM_CLASSES), fill_value=0)
    if (counts == 0).any():
        raise ValueError(f"학습 fold에 빠진 클래스가 있습니다: {counts.to_dict()}")
    return {int(i): float(len(rows) / (NUM_CLASSES * count)) for i, count in counts.items()}


def build_training_arrays(
    rows: pd.DataFrame,
    config: FamilyConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    noise_paths = rows.loc[rows["label"] == UNKNOWN_LABEL, "filepath"].tolist()
    class_weights = balanced_class_weights(rows) if config.class_weighting else {
        i: 1.0 for i in range(NUM_CLASSES)
    }
    variants = 1 + (AUGMENTATIONS_PER_FILE if config.augmentation == "light" else 0)
    features: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    weights: list[np.ndarray] = []

    for row in rows.itertuples(index=False):
        for augmentation_index in range(variants):
            embeddings = cached_features(row, augmentation_index, noise_paths)
            if config.representation == "clip_mean":
                current = embeddings.mean(axis=0, keepdims=True)
            else:
                current = embeddings
            count = len(current)
            features.append(current)
            labels.append(np.full(count, row.label_id, dtype=np.int64))
            # 원본 파일별 총 가중치를 같게 한 뒤 class/hard-negative 가중치를 반영합니다.
            hard_negative_multiplier = (
                config.hard_negative_weight
                if row.label_id == UNKNOWN_ID and bool(row.is_hard_negative)
                else 1.0
            )
            per_sample = class_weights[row.label_id] * hard_negative_multiplier / (variants * count)
            weights.append(np.full(count, per_sample, dtype=np.float32))

    x = np.concatenate(features).astype(np.float32)
    y = np.concatenate(labels).astype(np.int64)
    sample_weight = np.concatenate(weights).astype(np.float32)
    sample_weight *= len(sample_weight) / sample_weight.sum()
    return x, y, sample_weight


def build_validation_arrays(
    rows: pd.DataFrame,
    representation: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    features: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    weights: list[np.ndarray] = []
    for row in rows.itertuples(index=False):
        embeddings = cached_features(row)
        current = embeddings.mean(axis=0, keepdims=True) if representation == "clip_mean" else embeddings
        features.append(current)
        labels.append(np.full(len(current), row.label_id, dtype=np.int64))
        weights.append(np.full(len(current), 1.0 / len(current), dtype=np.float32))
    x = np.concatenate(features).astype(np.float32)
    y = np.concatenate(labels).astype(np.int64)
    sample_weight = np.concatenate(weights).astype(np.float32)
    sample_weight *= len(sample_weight) / sample_weight.sum()
    return x, y, sample_weight


def build_classifier(
    config: FamilyConfig,
    x_train: np.ndarray,
    seed: int,
    normalization_weights: np.ndarray | None = None,
) -> tf.keras.Model:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    inputs = tf.keras.layers.Input(shape=(1024,), dtype=tf.float32, name="embedding")

    if config.head == "baseline":
        x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(inputs)
        x = tf.keras.layers.Dropout(0.3, seed=seed, name="dropout_1")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
        x = tf.keras.layers.Dropout(0.3, seed=seed + 1, name="dropout_2")(x)
    else:
        if normalization_weights is None:
            normalization_weights = np.ones(len(x_train), dtype=np.float32)
        mean = np.average(x_train, axis=0, weights=normalization_weights)
        variance = np.average((x_train - mean) ** 2, axis=0, weights=normalization_weights) + 1e-6
        x = tf.keras.layers.Normalization(
            axis=-1,
            mean=mean,
            variance=variance,
            name="embedding_normalization",
        )(inputs)
        if config.head == "compact":
            x = tf.keras.layers.Dense(
                128,
                activation="swish",
                kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                name="compact_dense",
            )(x)
            x = tf.keras.layers.Dropout(0.25, seed=seed, name="compact_dropout")(x)
        elif config.head != "linear":
            raise ValueError(f"알 수 없는 head: {config.head}")

    output_regularizer = None if config.head == "baseline" else tf.keras.regularizers.l2(1e-4)
    logits = tf.keras.layers.Dense(NUM_CLASSES, kernel_regularizer=output_regularizer, name="logits")(x)
    model = tf.keras.Model(inputs, logits, name=f"hearo_{config.head}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def softmax_np(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    scaled = np.asarray(logits, np.float64) / max(float(temperature), 1e-4)
    scaled -= scaled.max(axis=-1, keepdims=True)
    exp = np.exp(scaled)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype(np.float64)


def pool_frame_logits(logits: np.ndarray, pooling: str, temperature: float = 1.0) -> np.ndarray:
    logits = np.asarray(logits, np.float64)
    if len(logits) == 1:
        return softmax_np(logits, temperature)[0]
    probabilities = softmax_np(logits, temperature)
    if pooling == "mean_probability":
        pooled = probabilities.mean(axis=0)
    elif pooling == "topk_probability":
        k = max(1, int(math.ceil(len(probabilities) * 0.5)))
        pooled = np.sort(probabilities, axis=0)[-k:].mean(axis=0)
    elif pooling == "logit_logmeanexp":
        # 배포 TFLite는 calibrated probability를 출력하므로 log(probability)에서 집계합니다.
        log_probabilities = np.log(np.clip(probabilities, 1e-12, 1.0))
        maximum = log_probabilities.max(axis=0, keepdims=True)
        pooled_logits = maximum[0] + np.log(
            np.exp(log_probabilities - maximum).mean(axis=0) + 1e-12
        )
        pooled = softmax_np(pooled_logits[None, :])[0]
    else:
        raise ValueError(f"알 수 없는 pooling: {pooling}")
    pooled = np.clip(pooled, 1e-12, None)
    return pooled / pooled.sum()


def aggregate_context(
    frame_logits: np.ndarray,
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
) -> np.ndarray:
    if context_frames == "full":
        return pool_frame_logits(frame_logits, pooling, temperature)
    width = max(1, int(context_frames))
    if len(frame_logits) <= width:
        windows = [frame_logits]
    else:
        windows = [frame_logits[start:start + width] for start in range(len(frame_logits) - width + 1)]
    window_probabilities = np.stack([
        pool_frame_logits(window, pooling, temperature) for window in windows
    ])
    # 실시간 시스템의 "어느 window에서든 알림" 조건을 파일 단위 평가에 반영합니다.
    target_confidence = window_probabilities[:, :UNKNOWN_ID].max(axis=1)
    return window_probabilities[int(np.argmax(target_confidence))]


def decision_predictions(probabilities: np.ndarray, thresholds: float | np.ndarray) -> np.ndarray:
    probabilities = np.asarray(probabilities)
    raw = probabilities.argmax(axis=1)
    confidence = probabilities[np.arange(len(probabilities)), raw]
    threshold_array = np.full(NUM_CLASSES, float(thresholds)) if np.isscalar(thresholds) else np.asarray(thresholds)
    accepted = (raw != UNKNOWN_ID) & (confidence >= threshold_array[raw])
    return np.where(accepted, raw, UNKNOWN_ID).astype(np.int64)


def expected_calibration_error(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = (prediction == y_true).astype(float)
    error = 0.0
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            error += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(error)


def operating_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    thresholds: float | np.ndarray,
) -> dict[str, float]:
    prediction = decision_predictions(probabilities, thresholds)
    unknown_mask = y_true == UNKNOWN_ID
    false_alert_rate = float(np.mean(prediction[unknown_mask] != UNKNOWN_ID)) if unknown_mask.any() else float("nan")
    knock_false_alert_rate = float(np.mean(np.isin(prediction[unknown_mask], KNOCK_IDS))) if unknown_mask.any() else float("nan")
    siren_false_alert_rate = float(np.mean(np.isin(prediction[unknown_mask], SIREN_IDS))) if unknown_mask.any() else float("nan")
    return {
        "target_macro_f1": float(f1_score(y_true, prediction, labels=range(UNKNOWN_ID), average="macro", zero_division=0)),
        "all_class_macro_f1": float(f1_score(y_true, prediction, labels=range(NUM_CLASSES), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "false_alert_rate": false_alert_rate,
        "unknown_to_knock_false_alert_rate": knock_false_alert_rate,
        "unknown_to_siren_false_alert_rate": siren_false_alert_rate,
        "ece": expected_calibration_error(y_true, probabilities),
    }


def tune_global_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, dict[str, float]]:
    best: tuple[float, float, float, dict[str, float]] | None = None
    for threshold in np.unique(np.concatenate([np.linspace(0.0, 0.995, 200), [1.0]])):
        metrics = operating_metrics(y_true, probabilities, float(threshold))
        feasible = metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12
        key = (float(feasible), metrics["target_macro_f1"], metrics["accuracy"])
        if best is None or key > best[:3]:
            best = (*key, {**metrics, "threshold": float(threshold)})
    assert best is not None
    return float(best[3]["threshold"]), best[3]


def tune_class_thresholds(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    initial_threshold: float,
) -> tuple[np.ndarray, dict[str, float]]:
    thresholds = np.full(NUM_CLASSES, initial_threshold, dtype=np.float64)
    thresholds[UNKNOWN_ID] = 0.0
    grid = np.linspace(0.05, 1.0, 96)
    for _ in range(2):
        for class_id in range(UNKNOWN_ID):
            best_threshold = thresholds[class_id]
            best_metrics = operating_metrics(y_true, probabilities, thresholds)
            best_key = (
                float(best_metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12),
                best_metrics["target_macro_f1"],
                best_metrics["accuracy"],
            )
            for candidate in grid:
                trial = thresholds.copy()
                trial[class_id] = candidate
                metrics = operating_metrics(y_true, probabilities, trial)
                key = (
                    float(metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12),
                    metrics["target_macro_f1"],
                    metrics["accuracy"],
                )
                if key > best_key:
                    best_key, best_threshold, best_metrics = key, float(candidate), metrics
            thresholds[class_id] = best_threshold
    final_metrics = operating_metrics(y_true, probabilities, thresholds)
    return thresholds, final_metrics


def fit_temperature(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    pseudo_logits = np.log(np.clip(probabilities, 1e-9, 1.0))

    def negative_log_likelihood(log_temperature: float) -> float:
        temperature = float(np.exp(log_temperature))
        calibrated = softmax_np(pseudo_logits, temperature)
        return float(-np.log(np.clip(calibrated[np.arange(len(y_true)), y_true], 1e-12, 1.0)).mean())

    result = minimize_scalar(negative_log_likelihood, bounds=(-3.0, 3.0), method="bounded")
    return float(np.exp(result.x)) if result.success else 1.0


def fit_record_temperature(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int,
) -> float:
    def negative_log_likelihood(log_temperature: float) -> float:
        temperature = float(np.exp(log_temperature))
        y_true, probabilities, _ = records_to_arrays(
            records, pooling, context_frames, temperature
        )
        return float(-np.log(
            np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-12, 1.0)
        ).mean())

    result = minimize_scalar(negative_log_likelihood, bounds=(-3.0, 3.0), method="bounded")
    return float(np.exp(result.x)) if result.success else 1.0



## 6. Inner CV 실행기



In [ ]:
POOLINGS = ["mean_probability", "topk_probability", "logit_logmeanexp"]


def train_fold(
    config: FamilyConfig,
    train_rows: pd.DataFrame,
    val_rows: pd.DataFrame,
    fold_number: int,
) -> tuple[tf.keras.Model, int, dict[str, list[float]]]:
    assert set(train_rows["group_id"]).isdisjoint(set(val_rows["group_id"]))
    x_train, y_train, w_train = build_training_arrays(train_rows, config)
    x_val, y_val, w_val = build_validation_arrays(val_rows, config.representation)
    model = build_classifier(config, x_train, ACTIVE_TRAINING_SEED + fold_number, w_train)
    history = model.fit(
        x_train,
        y_train,
        sample_weight=w_train,
        validation_data=(x_val, y_val, w_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, min_delta=1e-4
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=LR_PATIENCE, min_lr=1e-5
            ),
        ],
    )
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    return model, best_epoch, history.history


def predict_frame_records(
    model: tf.keras.Model,
    rows: pd.DataFrame,
    representation: str,
    fold_number: int,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    inputs: list[np.ndarray] = []
    row_metadata: list[tuple[Any, int]] = []
    chunk_frames = 0

    def flush_chunk() -> None:
        nonlocal inputs, row_metadata, chunk_frames
        if not inputs:
            return
        concatenated = np.concatenate(inputs).astype(np.float32)
        all_logits = model.predict(
            concatenated, batch_size=PREDICT_BATCH_SIZE, verbose=0
        ).astype(np.float32)
        offset = 0
        for queued_row, count in row_metadata:
            records.append({
                "relative_path": queued_row.relative_path,
                "group_id": queued_row.group_id,
                "label_id": int(queued_row.label_id),
                "fold": fold_number,
                "frame_logits": all_logits[offset:offset + count],
            })
            offset += count
        if offset != len(all_logits):
            raise AssertionError("배치 예측 결과 분할 길이가 일치하지 않습니다.")
        inputs = []
        row_metadata = []
        chunk_frames = 0

    for row in rows.itertuples(index=False):
        embeddings = cached_features(row)
        current = embeddings.mean(axis=0, keepdims=True) if representation == "clip_mean" else embeddings
        if inputs and chunk_frames + len(current) > PREDICT_CHUNK_FRAMES:
            flush_chunk()
        inputs.append(current)
        row_metadata.append((row, len(current)))
        chunk_frames += len(current)
    flush_chunk()
    return records


def records_to_arrays(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ordered = sorted(records, key=lambda item: item["relative_path"])
    y_true = np.asarray([item["label_id"] for item in ordered], dtype=np.int64)
    groups = np.asarray([item["group_id"] for item in ordered], dtype=object)
    probabilities = np.stack([
        aggregate_context(item["frame_logits"], pooling, context_frames, temperature)
        for item in ordered
    ])
    return y_true, probabilities, groups


def score_lifestyle_negative_candidates(
    records: list[dict[str, Any]],
    pooling: str,
    min_confidence: float = HARD_NEGATIVE_MIN_CONFIDENCE,
) -> pd.DataFrame:
    """Development OOF에서 비표적 후보의 표적 확률과 raw top-1 오류를 계산합니다."""
    ordered = sorted(records, key=lambda item: item["relative_path"])
    if any(int(item["fold"]) <= 0 for item in ordered):
        raise AssertionError("하드 네거티브 탐색에는 development OOF 예측만 사용할 수 있습니다.")
    paths = [item["relative_path"] for item in ordered]
    if len(paths) != len(set(paths)):
        raise AssertionError("OOF 하드 네거티브 예측 경로가 중복되었습니다.")
    y_true, probabilities, _ = records_to_arrays(ordered, pooling)
    raw_prediction = probabilities.argmax(axis=1)
    raw_confidence = probabilities[np.arange(len(probabilities)), raw_prediction]
    target_probabilities = probabilities[:, :UNKNOWN_ID]
    target_prediction = target_probabilities.argmax(axis=1)
    target_confidence = target_probabilities.max(axis=1)

    lookup = dev_df.set_index("relative_path")[[
        "hard_negative_candidate", "environment_category", "source_dataset", "license"
    ]].to_dict("index")
    rows: list[dict[str, Any]] = []
    for index, path in enumerate(paths):
        if y_true[index] != UNKNOWN_ID:
            continue
        info = lookup.get(path, {})
        curated = bool(info.get("hard_negative_candidate", False))
        is_raw_error = (
            raw_prediction[index] != UNKNOWN_ID
            and raw_confidence[index] >= min_confidence
        )
        if not curated and not is_raw_error:
            continue
        rows.append({
            "relative_path": path,
            "true_label": UNKNOWN_LABEL,
            "confused_with": CATEGORIES[int(target_prediction[index])],
            "target_probability": float(target_confidence[index]),
            "confidence": float(target_confidence[index]),
            "raw_prediction": CATEGORIES[int(raw_prediction[index])],
            "raw_prediction_confidence": float(raw_confidence[index]),
            "environment_category": str(info.get("environment_category", "기존_비표적음")) or "기존_비표적음",
            "source_dataset": str(info.get("source_dataset", "")),
            "license": str(info.get("license", "")),
            "is_true_hard_negative": bool(is_raw_error),
            "is_semihard_p010": bool((not is_raw_error) and target_confidence[index] >= SEMI_HARD_MIN_TARGET_PROBABILITY),
        })
    candidates = pd.DataFrame(rows)
    if candidates.empty:
        raise RuntimeError("Development OOF 생활소음 후보가 없습니다. 공개 생활소음 준비 단계를 확인하세요.")
    return candidates.sort_values(
        ["is_true_hard_negative", "target_probability"], ascending=[False, False]
    ).reset_index(drop=True)


def select_quota_hard_negatives(
    candidates: pd.DataFrame,
    min_total: int = MIN_HARD_NEGATIVES_TOTAL,
    min_per_environment_category: int = MIN_HARD_NEGATIVES_PER_ENV_CATEGORY,
) -> pd.DataFrame:
    """v2.2 비교군: 실제 오류를 모두 포함하고 환경별/전체 할당량을 채웁니다."""
    selected_paths = set(
        candidates.loc[candidates["is_true_hard_negative"], "relative_path"].astype(str)
    )
    open_categories = sorted(OPEN_HARD_NEGATIVE_QUERIES)
    available_categories = set(candidates["environment_category"].astype(str))
    missing_categories = sorted(set(open_categories) - available_categories)
    if missing_categories:
        raise RuntimeError(
            "Development split에 공개 생활소음 범주가 없습니다: "
            f"{missing_categories}. 범주별 독립 uploader/group을 더 추가하세요."
        )
    for category in open_categories:
        category_rows = candidates[candidates["environment_category"] == category]
        already = int(category_rows["relative_path"].isin(selected_paths).sum())
        needed = max(0, min_per_environment_category - already)
        additions = category_rows[
            ~category_rows["relative_path"].isin(selected_paths)
        ].head(needed)
        selected_paths.update(additions["relative_path"].astype(str))

    if len(selected_paths) < min_total:
        additions = candidates[
            ~candidates["relative_path"].isin(selected_paths)
        ].head(min_total - len(selected_paths))
        selected_paths.update(additions["relative_path"].astype(str))

    selected = candidates[candidates["relative_path"].isin(selected_paths)].copy()
    if len(selected) < min_total:
        raise RuntimeError(
            f"Development OOF hard/semi-hard negative가 {len(selected)}개뿐입니다. "
            f"최소 {min_total}개를 위해 CC0 생활소음을 더 추가하세요."
        )
    per_category = selected.groupby("environment_category").size()
    missing_quota = {
        category: int(min_per_environment_category - per_category.get(category, 0))
        for category in open_categories
        if per_category.get(category, 0) < min_per_environment_category
    }
    if missing_quota:
        raise RuntimeError(f"카테고리별 hard/semi-hard negative 최소량 부족: {missing_quota}")
    return selected.sort_values(
        ["is_true_hard_negative", "target_probability"], ascending=[False, False]
    ).reset_index(drop=True)


def select_v3_hard_negative_arms(
    candidates: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    """사전 정의된 네 비교군의 학습 파일을 OOF 점수만으로 확정합니다."""
    raw = candidates[candidates["is_true_hard_negative"]].copy()
    if raw.empty:
        raise RuntimeError("실제 raw top-1 하드 네거티브가 없습니다.")
    semi = candidates[
        candidates["is_true_hard_negative"]
        | (candidates["target_probability"] >= SEMI_HARD_MIN_TARGET_PROBABILITY)
    ].copy()
    quota = select_quota_hard_negatives(candidates)
    empty = candidates.iloc[0:0].copy()
    selections = {
        "baseline": empty,
        "raw_top1_2x": raw,
        "raw_plus_semihard_p010_1p5x": semi,
        "quota60_2x": quota,
    }

    raw_paths = set(raw["relative_path"].astype(str))
    semi_paths = set(semi["relative_path"].astype(str))
    quota_paths = set(quota["relative_path"].astype(str))
    if not raw_paths.issubset(semi_paths) or not raw_paths.issubset(quota_paths):
        raise AssertionError("실제 raw top-1 오류가 보강 비교군에서 누락되었습니다.")
    semi_only = semi[~semi["is_true_hard_negative"]]
    if not semi_only.empty and (
        semi_only["target_probability"] < SEMI_HARD_MIN_TARGET_PROBABILITY - 1e-12
    ).any():
        raise AssertionError("0.10 미만의 쉬운 표본이 semi-hard 비교군에 포함되었습니다.")
    if len(quota) < MIN_HARD_NEGATIVES_TOTAL:
        raise AssertionError("할당량 비교군이 최소 60개를 충족하지 못했습니다.")

    for arm_id, frame in selections.items():
        frame["arm_id"] = arm_id
        frame["selection_policy"] = V3_ARM_SPECS[arm_id]["selection_policy"]
        frame["training_weight"] = V3_ARM_SPECS[arm_id]["hard_negative_weight"]
        frame["selection_source"] = np.where(
            frame.get("is_true_hard_negative", pd.Series(False, index=frame.index)),
            "development_oof_raw_top1",
            "development_oof_semihard_or_quota",
        )
    return selections


def build_v3_hard_negative_selections(
    records: list[dict[str, Any]],
    pooling: str,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    candidates = score_lifestyle_negative_candidates(records, pooling)
    return candidates, select_v3_hard_negative_arms(candidates)


def activate_hard_negative_selection(
    paths: Iterable[str],
    *,
    include_metadata: bool = False,
) -> None:
    """한 실험 팔의 경로만 활성화해 팔 사이 가중치 혼입을 방지합니다."""
    selected = set(map(str, paths))
    dev_df["is_hard_negative"] = dev_df["relative_path"].astype(str).isin(selected)
    test_df["is_hard_negative"] = False
    if include_metadata:
        metadata["is_hard_negative"] = metadata["relative_path"].astype(str).isin(selected)


def mine_lifestyle_hard_negatives(
    records: list[dict[str, Any]],
    pooling: str,
    min_confidence: float = HARD_NEGATIVE_MIN_CONFIDENCE,
    min_total: int = MIN_HARD_NEGATIVES_TOTAL,
    min_per_environment_category: int = MIN_HARD_NEGATIVES_PER_ENV_CATEGORY,
) -> pd.DataFrame:
    """하위 호환용 v2.2 할당량 비교군 선택 함수."""
    candidates = score_lifestyle_negative_candidates(records, pooling, min_confidence)
    return select_quota_hard_negatives(candidates, min_total, min_per_environment_category)


def summarize_records(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
    threshold_mode: str = "global",
) -> dict[str, Any]:
    y_true, probabilities, groups = records_to_arrays(records, pooling, context_frames, temperature)
    global_threshold, metrics = tune_global_threshold(y_true, probabilities)
    thresholds: float | np.ndarray = global_threshold
    if threshold_mode == "per_class":
        thresholds, metrics = tune_class_thresholds(y_true, probabilities, global_threshold)
    ordered = sorted(records, key=lambda item: item["relative_path"])
    fold_ids = np.asarray([item["fold"] for item in ordered], dtype=np.int64)
    fold_metrics = [
        operating_metrics(y_true[fold_ids == fold], probabilities[fold_ids == fold], thresholds)
        for fold in np.unique(fold_ids)
    ]
    metrics["cv_target_macro_f1_mean"] = float(np.mean([
        item["target_macro_f1"] for item in fold_metrics
    ]))
    metrics["cv_target_macro_f1_std"] = float(np.std([
        item["target_macro_f1"] for item in fold_metrics
    ]))
    metrics["cv_false_alert_rate_mean"] = float(np.mean([
        item["false_alert_rate"] for item in fold_metrics if np.isfinite(item["false_alert_rate"])
    ]))
    return {
        "pooling": pooling,
        "context_frames": context_frames,
        "temperature": float(temperature),
        "threshold_mode": threshold_mode,
        "thresholds": thresholds,
        "metrics": metrics,
        "y_true": y_true,
        "probabilities": probabilities,
        "groups": groups,
    }


def cv_cache_path(config: FamilyConfig) -> Path:
    hard_negative_paths = sorted(
        dev_df.loc[dev_df["is_hard_negative"], "relative_path"].astype(str)
    )
    manifest_rows = metadata[["relative_path", "label", "group_id", "audio_hash"]].sort_values(
        "relative_path"
    ).to_dict("records")
    payload = {
        "pipeline_revision": OOF_CACHE_REVISION,
        "config": asdict(config),
        "manifest": manifest_rows,
        "inner_seed": inner_seed,
        "inner_splits": inner_splits_count,
        "hard_negative_paths": hard_negative_paths,
        "training_parameters": {
            "seed": ACTIVE_TRAINING_SEED,
            "target_sample_rate": TARGET_SR,
            "yamnet_handle": YAMNET_HANDLE,
            "run_mode": RUN_MODE,
            "inner_folds": INNER_FOLDS,
            "max_epochs": MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "predict_batch_size": PREDICT_BATCH_SIZE,
            "predict_chunk_frames": PREDICT_CHUNK_FRAMES,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "lr_patience": LR_PATIENCE,
            "augmentations_per_file": AUGMENTATIONS_PER_FILE,
            "max_false_alert_rate": MAX_FALSE_ALERT_RATE,
            "hard_negative_min_confidence": HARD_NEGATIVE_MIN_CONFIDENCE,
            "min_hard_negatives_total": MIN_HARD_NEGATIVES_TOTAL,
            "min_hard_negatives_per_environment_category": MIN_HARD_NEGATIVES_PER_ENV_CATEGORY,
        },
    }
    digest = hashlib.sha256(
        json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]
    safe_name = "".join(character if character.isalnum() or character in "-_" else "_" for character in config.name)
    return CV_CACHE_DIR / f"{safe_name}-{digest}.pkl"


def run_cv_family(config: FamilyConfig) -> dict[str, Any]:
    cache_path = cv_cache_path(config)
    if cache_path.exists():
        # 이 pickle은 현재 파이프라인이 직접 생성한 신뢰 가능한 캐시만 읽습니다.
        with cache_path.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("config") == config:
            print(f"\n[CV 캐시 재사용] {config.name}: {cache_path.name}")
            return cached

    print(f"\n[CV] {config}")
    started = time.time()
    records: list[dict[str, Any]] = []
    best_epochs: list[int] = []
    histories = []
    for fold_number, (train_idx, val_idx) in enumerate(inner_splits, start=1):
        fold_cache = cache_path.with_name(f"{cache_path.stem}-fold{fold_number}.pkl")
        if fold_cache.exists():
            with fold_cache.open("rb") as file:
                fold_result = pickle.load(file)
            print(f"  fold {fold_number}/{len(inner_splits)} 캐시 재사용")
        else:
            train_rows = dev_df.iloc[train_idx].reset_index(drop=True)
            val_rows = dev_df.iloc[val_idx].reset_index(drop=True)
            model, best_epoch, history = train_fold(config, train_rows, val_rows, fold_number)
            fold_result = {
                "records": predict_frame_records(model, val_rows, config.representation, fold_number),
                "best_epoch": best_epoch,
                "history": history,
            }
            fold_temporary = fold_cache.with_suffix(".tmp")
            with fold_temporary.open("wb") as file:
                pickle.dump(fold_result, file, protocol=pickle.HIGHEST_PROTOCOL)
            fold_temporary.replace(fold_cache)
            print(f"  fold {fold_number}/{len(inner_splits)}: best epoch={best_epoch}")
            del model
            tf.keras.backend.clear_session()
        records.extend(fold_result["records"])
        best_epochs.append(int(fold_result["best_epoch"]))
        histories.append(fold_result["history"])

    available_poolings = ["mean_probability"] if config.representation == "clip_mean" else POOLINGS
    summaries = {
        pooling: summarize_records(records, pooling)
        for pooling in available_poolings
    }
    elapsed = time.time() - started
    result = {
        "config": config,
        "records": records,
        "best_epochs": best_epochs,
        "histories": histories,
        "summaries": summaries,
        "elapsed_seconds": elapsed,
    }
    temporary = cache_path.with_suffix(".tmp")
    with temporary.open("wb") as file:
        pickle.dump(result, file, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(cache_path)
    print(f"  CV 결과 캐시 저장: {cache_path.name}")
    return result


def candidate_key(candidate: dict[str, Any]) -> tuple[float, float, float]:
    metrics = candidate["summary"]["metrics"]
    feasible = metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12
    return float(feasible), metrics["cv_target_macro_f1_mean"], metrics["accuracy"]


def candidate_from_family(family_result: dict[str, Any], pooling: str) -> dict[str, Any]:
    return {
        "family_result": family_result,
        "summary": family_result["summaries"][pooling],
    }


experiment_rows: list[dict[str, Any]] = []


def log_candidate(round_number: int, candidate: dict[str, Any], promoted: bool = False) -> None:
    config = candidate["family_result"]["config"]
    summary = candidate["summary"]
    metrics = summary["metrics"]
    experiment_rows.append({
        "round": round_number,
        "family": config.name,
        "head": config.head,
        "representation": config.representation,
        "augmentation": config.augmentation,
        "class_weighting": config.class_weighting,
        "hard_negative_weight": config.hard_negative_weight,
        "arm_id": candidate.get("arm_id", config.name),
        "selection_policy": candidate.get("selection_policy", "unspecified"),
        "selected_negative_files": len(candidate.get("hard_negative_paths", ())),
        "selected_true_hard_negatives": int(candidate.get("selected_true_hard_negatives", 0)),
        "selected_semihard_negatives": int(candidate.get("selected_semihard_negatives", 0)),
        "pooling": summary["pooling"],
        "context_frames": summary["context_frames"],
        "temperature": summary["temperature"],
        "threshold_mode": summary["threshold_mode"],
        "family_cv_elapsed_seconds": candidate["family_result"]["elapsed_seconds"],
        **metrics,
        "promoted": promoted,
    })


def improvement_over(challenger: dict[str, Any], incumbent: dict[str, Any]) -> float:
    return (
        challenger["summary"]["metrics"]["cv_target_macro_f1_mean"]
        - incumbent["summary"]["metrics"]["cv_target_macro_f1_mean"]
    )



## 7. 핵심 실험: 기준선 → 하드 네거티브 → 클래스별 임계값


In [ ]:
# 사전 등록된 네 비교군을 development grouped OOF에서만 비교합니다.
activate_hard_negative_selection(())
baseline_family = run_cv_family(FamilyConfig(
    name="v3_baseline",
    head="baseline",
    representation="clip_mean",
))
baseline = candidate_from_family(baseline_family, "mean_probability")

hard_negative_candidates_table, v3_hard_negative_selections = build_v3_hard_negative_selections(
    baseline_family["records"], baseline["summary"]["pooling"]
)
hard_negative_candidates_table.to_csv(
    OUTPUT_DIR / "hard_negative_candidates_v3.csv", index=False, encoding="utf-8-sig"
)

v3_candidates: dict[str, dict[str, Any]] = {}
for arm_id, spec in V3_ARM_SPECS.items():
    selected = v3_hard_negative_selections[arm_id]
    selected_paths = tuple(sorted(selected["relative_path"].astype(str)))
    activate_hard_negative_selection(selected_paths)
    if arm_id == "baseline":
        family = baseline_family
        candidate = baseline
    else:
        family = run_cv_family(FamilyConfig(
            name=f"v3_{arm_id}",
            head="baseline",
            representation="clip_mean",
            hard_negative_weight=float(spec["hard_negative_weight"]),
        ))
        candidate = candidate_from_family(family, "mean_probability")
    selected_true = int(selected.get("is_true_hard_negative", pd.Series(dtype=bool)).sum())
    candidate.update({
        "arm_id": arm_id,
        "display_name": spec["display_name"],
        "selection_policy": spec["selection_policy"],
        "hard_negative_paths": selected_paths,
        "selected_true_hard_negatives": selected_true,
        "selected_semihard_negatives": int(len(selected) - selected_true),
    })
    v3_candidates[arm_id] = candidate

arm_membership_frames = [
    frame.copy()
    for arm_id, frame in v3_hard_negative_selections.items()
    if arm_id != "baseline"
]
hard_negative_arm_membership = pd.concat(arm_membership_frames, ignore_index=True)
hard_negative_arm_membership.to_csv(
    OUTPUT_DIR / "hard_negative_arm_membership_v3.csv", index=False, encoding="utf-8-sig"
)

# candidate_key는 development OOF false-alert 제약과 F1만 사용합니다. Test는 아직 호출하지 않습니다.
selected_arm_id, champion = max(v3_candidates.items(), key=lambda item: candidate_key(item[1]))
round1_promoted = selected_arm_id != "baseline"
for arm_id, candidate in v3_candidates.items():
    log_candidate(1, candidate, promoted=(arm_id == selected_arm_id))

hard_negatives = v3_hard_negative_selections[selected_arm_id].copy()
hard_negative_paths = set(champion["hard_negative_paths"])
activate_hard_negative_selection(hard_negative_paths, include_metadata=True)
hard_negatives.to_csv(OUTPUT_DIR / "hard_negatives.csv", index=False, encoding="utf-8-sig")

selection_summary_rows = []
for arm_id, candidate in v3_candidates.items():
    metrics = candidate["summary"]["metrics"]
    selection_summary_rows.append({
        "arm_id": arm_id,
        "display_name": candidate["display_name"],
        "selection_policy": candidate["selection_policy"],
        "hard_negative_weight": candidate["family_result"]["config"].hard_negative_weight,
        "selected_negative_files": len(candidate["hard_negative_paths"]),
        "selected_true_hard_negatives": candidate["selected_true_hard_negatives"],
        "selected_semihard_negatives": candidate["selected_semihard_negatives"],
        "development_oof_target_macro_f1": metrics["cv_target_macro_f1_mean"],
        "development_oof_false_alert_rate": metrics["false_alert_rate"],
        "development_selected": arm_id == selected_arm_id,
    })
v3_development_arm_comparison = pd.DataFrame(selection_summary_rows)
v3_development_arm_comparison.to_csv(
    OUTPUT_DIR / "development_oof_arm_comparison_v3.csv", index=False, encoding="utf-8-sig"
)
display(v3_development_arm_comparison)
print("Development OOF 선택 비교군:", selected_arm_id)
print("실제 raw top-1 오분류 수:", int(hard_negative_candidates_table["is_true_hard_negative"].sum()))


In [ ]:
# 2단계: 같은 champion OOF 확률에서 전역 임계값과 클래스별 임계값만 비교합니다.
incumbent_before_threshold = champion
base_summary = incumbent_before_threshold["summary"]
per_class_summary = summarize_records(
    incumbent_before_threshold["family_result"]["records"],
    base_summary["pooling"],
    base_summary["context_frames"],
    base_summary["temperature"],
    threshold_mode="per_class",
)
per_class_candidate = {
    **incumbent_before_threshold,
    "summary": per_class_summary,
}
round2_promoted = (
    candidate_key(per_class_candidate)[0] == 1.0
    and improvement_over(per_class_candidate, incumbent_before_threshold) >= MIN_PROMOTION
)
if round2_promoted:
    champion = per_class_candidate
log_candidate(2, per_class_candidate, promoted=round2_promoted)
print("클래스별 임계값 승격:", round2_promoted)


In [ ]:
rounds_completed = 2
round3_promoted = False
experiment_results = pd.DataFrame(experiment_rows)
experiment_results.to_csv(
    OUTPUT_DIR / "experiment_results.csv", index=False, encoding="utf-8-sig"
)
experiment_results[experiment_results["round"] == 1].to_csv(
    OUTPUT_DIR / "development_oof_arm_metrics_v3.csv", index=False, encoding="utf-8-sig"
)
display(experiment_results.sort_values(["round", "cv_target_macro_f1_mean"], ascending=[True, False]))

champion_config: FamilyConfig = champion["family_result"]["config"]
champion_summary = champion["summary"]
print("\n최종 선택 구성")
print("  run mode:", RUN_MODE)
print("  config:", champion_config)
print("  pooling:", champion_summary["pooling"])
print("  threshold mode:", champion_summary["threshold_mode"])
print("  development metrics:", champion_summary["metrics"])


## 8. Untouched test 1회 평가



In [ ]:
def train_fixed_epochs(
    rows: pd.DataFrame,
    config: FamilyConfig,
    epochs: int,
    seed: int,
) -> tuple[tf.keras.Model, dict[str, list[float]]]:
    x_train, y_train, sample_weight = build_training_arrays(rows, config)
    model = build_classifier(config, x_train, seed, sample_weight)
    history = model.fit(
        x_train,
        y_train,
        sample_weight=sample_weight,
        epochs=max(1, int(epochs)),
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    return model, history.history


representative_final_training_seeds = final_training_seeds(SEED)
PRIMARY_FINAL_TRAINING_SEED = representative_final_training_seeds[0]
activate_hard_negative_selection(champion.get("hard_negative_paths", ()))
selected_epochs = max(1, int(round(np.median(champion["family_result"]["best_epochs"]))))
test_model, dev_fit_history = train_fixed_epochs(
    dev_df, champion_config, selected_epochs, PRIMARY_FINAL_TRAINING_SEED
)
test_records = predict_frame_records(test_model, test_df, champion_config.representation, fold_number=-1)
test_y, test_probabilities, test_groups = records_to_arrays(
    test_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    champion_summary["temperature"],
)
test_thresholds = champion_summary["thresholds"]
test_metrics = operating_metrics(test_y, test_probabilities, test_thresholds)
test_predictions = decision_predictions(test_probabilities, test_thresholds)


def grouped_bootstrap_interval(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    groups: np.ndarray,
    thresholds: float | np.ndarray,
    iterations: int = BOOTSTRAP_ITERATIONS,
) -> dict[str, list[float]]:
    unique_groups = np.unique(groups)
    group_indices = {group: np.flatnonzero(groups == group) for group in unique_groups}
    rng = np.random.default_rng(SEED + 999)
    values = {"target_macro_f1": [], "false_alert_rate": [], "accuracy": []}
    for _ in range(iterations):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        metrics = operating_metrics(y_true[indices], probabilities[indices], thresholds)
        for key in values:
            if np.isfinite(metrics[key]):
                values[key].append(metrics[key])
    return {
        key: [float(np.percentile(series, 2.5)), float(np.percentile(series, 97.5))]
        for key, series in values.items() if series
    }


test_confidence_intervals = grouped_bootstrap_interval(
    test_y, test_probabilities, test_groups, test_thresholds
)

# 동일한 untouched test 호출에서 사전 선언된 기존 구조 기준선도 함께 평가합니다.
if champion_config == baseline["family_result"]["config"]:
    baseline_test_metrics = dict(test_metrics)
else:
    baseline_epochs = max(1, int(round(np.median(baseline["family_result"]["best_epochs"]))))
    activate_hard_negative_selection(baseline.get("hard_negative_paths", ()))
    baseline_test_model, _ = train_fixed_epochs(
        dev_df, baseline["family_result"]["config"], baseline_epochs, PRIMARY_FINAL_TRAINING_SEED
    )
    baseline_test_records = predict_frame_records(
        baseline_test_model,
        test_df,
        baseline["family_result"]["config"].representation,
        fold_number=-4,
    )
    baseline_test_y, baseline_test_probabilities, _ = records_to_arrays(
        baseline_test_records,
        baseline["summary"]["pooling"],
        baseline["summary"]["context_frames"],
        baseline["summary"]["temperature"],
    )
    baseline_test_metrics = operating_metrics(
        baseline_test_y,
        baseline_test_probabilities,
        baseline["summary"]["thresholds"],
    )
    del baseline_test_model
    tf.keras.backend.clear_session()

activate_hard_negative_selection(champion.get("hard_negative_paths", ()))
print("대표 split 공정 초기화 seed:", PRIMARY_FINAL_TRAINING_SEED)
print("Test metrics:", test_metrics)
print("Baseline test metrics:", baseline_test_metrics)
print("95% grouped bootstrap CI:", test_confidence_intervals)
print(classification_report(
    test_y,
    test_predictions,
    labels=range(NUM_CLASSES),
    target_names=CATEGORIES,
    zero_division=0,
))



## 9. 전체 데이터 재학습 및 TFLite 내보내기



In [ ]:
activate_hard_negative_selection(champion.get("hard_negative_paths", ()), include_metadata=True)
final_model, final_history = train_fixed_epochs(
    metadata, champion_config, selected_epochs, PRIMARY_FINAL_TRAINING_SEED
)

temperature_layer = tf.keras.layers.Rescaling(
    scale=1.0 / max(float(champion_summary["temperature"]), 1e-4),
    name="temperature_scaling",
)(final_model.output)
probability_output = tf.keras.layers.Softmax(name="probabilities")(temperature_layer)
deployment_model = tf.keras.Model(
    final_model.input,
    probability_output,
    name="hearo_classifier_v3_2",
)
deployment_model.save(MODEL_DIR / "hearo_classifier_v3_2.keras")


def convert_tflite(model: tf.keras.Model, dynamic_range: bool) -> bytes:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if dynamic_range:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()


float_tflite = convert_tflite(deployment_model, dynamic_range=False)
dynamic_tflite = convert_tflite(deployment_model, dynamic_range=True)
(MODEL_DIR / "hearo_classifier_v3_2_float32.tflite").write_bytes(float_tflite)
(MODEL_DIR / "hearo_classifier_v3_2_dynamic.tflite").write_bytes(dynamic_tflite)


def tflite_predict(interpreter: tf.lite.Interpreter, embeddings: np.ndarray) -> np.ndarray:
    input_details = interpreter.get_input_details()[0]
    requested_shape = [len(embeddings), 1024]
    if list(input_details["shape"]) != requested_shape:
        interpreter.resize_tensor_input(input_details["index"], requested_shape, strict=False)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()[0]
    interpreter.set_tensor(input_details["index"], embeddings.astype(input_details["dtype"]))
    interpreter.invoke()
    output_details = interpreter.get_output_details()[0]
    output = interpreter.get_tensor(output_details["index"])
    if np.issubdtype(output_details["dtype"], np.integer):
        scale, zero_point = output_details["quantization"]
        output = (output.astype(np.float32) - zero_point) * scale
    return np.asarray(output, np.float32)


def tflite_records(model_content: bytes, rows: pd.DataFrame) -> list[dict[str, Any]]:
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    records = []
    for row in rows.itertuples(index=False):
        embeddings = cached_features(row)
        model_input = embeddings.mean(axis=0, keepdims=True) if champion_config.representation == "clip_mean" else embeddings
        probabilities = np.clip(tflite_predict(interpreter, model_input), 1e-9, 1.0)
        records.append({
            "relative_path": row.relative_path,
            "group_id": row.group_id,
            "label_id": int(row.label_id),
            "fold": -2,
            # aggregate_context는 logits 입력이므로 log(probability)를 사용합니다.
            "frame_logits": np.log(probabilities),
        })
    return records


def evaluate_tflite(model_content: bytes) -> tuple[dict[str, float], np.ndarray]:
    records = tflite_records(model_content, test_df)
    y_true, probabilities, _ = records_to_arrays(
        records,
        champion_summary["pooling"],
        champion_summary["context_frames"],
        temperature=1.0,
    )
    return operating_metrics(y_true, probabilities, test_thresholds), probabilities


float_metrics, float_probabilities = evaluate_tflite(float_tflite)
dynamic_metrics, dynamic_probabilities = evaluate_tflite(dynamic_tflite)
quantized_drop = float_metrics["target_macro_f1"] - dynamic_metrics["target_macro_f1"]

keras_final_records = []
for row in test_df.itertuples(index=False):
    embeddings = cached_features(row)
    model_input = embeddings.mean(axis=0, keepdims=True) if champion_config.representation == "clip_mean" else embeddings
    probabilities = deployment_model.predict(model_input, verbose=0)
    keras_final_records.append({
        "relative_path": row.relative_path,
        "group_id": row.group_id,
        "label_id": int(row.label_id),
        "fold": -3,
        "frame_logits": np.log(np.clip(probabilities, 1e-9, 1.0)),
    })
_, keras_final_probabilities, _ = records_to_arrays(
    keras_final_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    temperature=1.0,
)
float_max_abs_error = float(np.max(np.abs(keras_final_probabilities - float_probabilities)))
dynamic_max_abs_error = float(np.max(np.abs(keras_final_probabilities - dynamic_probabilities)))
if float_max_abs_error > 1e-4:
    raise RuntimeError(
        f"Keras/float32 TFLite 확률 오차가 허용값을 초과했습니다: {float_max_abs_error:.6f}"
    )
use_dynamic = quantized_drop <= 0.005 and dynamic_max_abs_error <= 0.02
selected_tflite = dynamic_tflite if use_dynamic else float_tflite
(MODEL_DIR / "hearo_classifier_v3_2.tflite").write_bytes(selected_tflite)
selected_probabilities = dynamic_probabilities if use_dynamic else float_probabilities
tflite_max_abs_error = float(np.max(np.abs(keras_final_probabilities - selected_probabilities)))


def benchmark_tflite(model_content: bytes, example: np.ndarray, repeats: int = 100) -> float:
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    for _ in range(5):
        tflite_predict(interpreter, example)
    started = time.perf_counter()
    for _ in range(repeats):
        tflite_predict(interpreter, example)
    return float((time.perf_counter() - started) * 1_000.0 / repeats)


benchmark_row = next(test_df.itertuples(index=False))
benchmark_embeddings = cached_features(benchmark_row)
benchmark_input = (
    benchmark_embeddings.mean(axis=0, keepdims=True)
    if champion_config.representation == "clip_mean"
    else benchmark_embeddings
)
float_inference_ms = benchmark_tflite(float_tflite, benchmark_input)
dynamic_inference_ms = benchmark_tflite(dynamic_tflite, benchmark_input)

threshold_values = (
    np.full(NUM_CLASSES, float(test_thresholds))
    if np.isscalar(test_thresholds)
    else np.asarray(test_thresholds, dtype=float)
)
threshold_values[UNKNOWN_ID] = 0.0

with (MODEL_DIR / "categories_v3_2.txt").open("w", encoding="utf-8") as file:
    file.write("\n".join(CATEGORIES) + "\n")

model_metadata = {
    "schema_version": 4,
    "training_run_mode": RUN_MODE,
    "pipeline_revision": PIPELINE_REVISION,
    "model_name": "hearo_classifier_v3_2",
    "taxonomy_version": "hearo-8-target-plus-unknown-v2-siren-patterns",
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "final_training_seed_policy": "outer_split_seed + [1000, 2000, 3000], shared across methods",
    "deployment_final_training_seed": PRIMARY_FINAL_TRAINING_SEED,
    "test_seed_selection_used": False,
    "sample_rate": TARGET_SR,
    "record_step_seconds": 1.0,
    "rolling_buffer_seconds": 2.0,
    "yamnet_handle": YAMNET_HANDLE,
    "yamnet_frame_seconds": YAMNET_FRAME_SECONDS,
    "yamnet_hop_seconds": 0.48,
    "classifier_input": {"dtype": "float32", "shape": [None, 1024]},
    "classifier_output": {"dtype": "float32", "shape": [None, NUM_CLASSES], "type": "probabilities"},
    "categories": CATEGORIES,
    "unknown_label": UNKNOWN_LABEL,
    "representation": champion_config.representation,
    "frame_pooling": champion_summary["pooling"],
    "context_frames": champion_summary["context_frames"],
    "temperature": float(champion_summary["temperature"]),
    "temperature_embedded_in_tflite": True,
    "threshold_mode": champion_summary["threshold_mode"],
    "class_thresholds": {
        label: float(threshold_values[index]) for index, label in enumerate(CATEGORIES)
    },
    "unknown_false_alert_limit": MAX_FALSE_ALERT_RATE,
    "legacy_relabel": LEGACY_RELABEL,
    "hard_negative_policy": {
        "enabled": champion_config.hard_negative_weight > 1.0,
        "scope": "lifestyle_environment_all_target_classes",
        "categories": TARGET_CATEGORIES,
        "selected_arm": champion.get("arm_id", "baseline"),
        "selection_source": champion.get("selection_policy", "none"),
        "raw_top1_minimum_confidence": HARD_NEGATIVE_MIN_CONFIDENCE,
        "semihard_minimum_target_probability": SEMI_HARD_MIN_TARGET_PROBABILITY,
        "selected_files": len(champion.get("hard_negative_paths", ())),
        "training_weight": champion_config.hard_negative_weight,
        "siren_hard_negative_enabled": True,
    },
    "yamnet_gate": {
        "enabled": False,
        "speech_score_used": False,
        "reason": "YAMNet Speech 점수를 거부 근거로 사용하지 않고 비표적음 확률과 임계값만 사용",
    },
    "selected_tflite_variant": "dynamic_range" if use_dynamic else "float32",
    "selected_epochs": selected_epochs,
    "rounds_completed": rounds_completed,
    "class_mapping": {
        "노크_목재": "노크소리",
        "노크_철재문": "노크소리",
        "도어락_개방음": "도어락소리",
        "도어락_입력음": "도어락소리",
        "사이렌_고저교번형": "비상벨소리",
        "사이렌_완만변조형": "비상벨소리",
        "사이렌_급속변조형": "비상벨소리",
        "아기 울음": "아기울음소리",
    },
    "siren_label_definitions": SIREN_LABEL_DEFINITIONS,
}
(MODEL_DIR / "model_metadata_v3_2.json").write_text(
    json.dumps(model_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)



## 10. 결과 리포트와 시각화



In [ ]:
def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, FamilyConfig):
        return asdict(value)
    return value


metrics_report = {
    "selection_data": "development grouped out-of-fold",
    "representative_final_training_seeds": representative_final_training_seeds,
    "primary_final_training_seed": PRIMARY_FINAL_TRAINING_SEED,
    "test_seed_selection_used": False,
    "baseline_development": baseline["summary"]["metrics"],
    "baseline_untouched_test": baseline_test_metrics,
    "champion_development": champion_summary["metrics"],
    "untouched_test": test_metrics,
    "untouched_test_group_bootstrap_95ci": test_confidence_intervals,
    "champion_config": champion_config,
    "champion_decision": {
        "pooling": champion_summary["pooling"],
        "context_frames": champion_summary["context_frames"],
        "temperature": champion_summary["temperature"],
        "threshold_mode": champion_summary["threshold_mode"],
        "thresholds": test_thresholds,
    },
    "rounds_completed": rounds_completed,
    "promotion_delta": MIN_PROMOTION,
    "false_alert_limit": MAX_FALSE_ALERT_RATE,
    "legacy_relabel": LEGACY_RELABEL,
    "hard_negative_policy": {
        "scope": "lifestyle_environment_all_target_classes",
        "selected_arm": champion.get("arm_id", "baseline"),
        "selection_source": champion.get("selection_policy", "none"),
        "raw_top1_minimum_confidence": HARD_NEGATIVE_MIN_CONFIDENCE,
        "semihard_minimum_target_probability": SEMI_HARD_MIN_TARGET_PROBABILITY,
        "selected_files": len(champion.get("hard_negative_paths", ())),
        "training_weight": champion_config.hard_negative_weight,
    },
    "yamnet_speech_rejection": {"enabled": False},
    "tflite": {
        "float32_metrics_on_parity_set": float_metrics,
        "dynamic_metrics_on_parity_set": dynamic_metrics,
        "selected": "dynamic_range" if use_dynamic else "float32",
        "float32_bytes": len(float_tflite),
        "dynamic_bytes": len(dynamic_tflite),
        "selected_bytes": len(selected_tflite),
        "keras_tflite_max_abs_probability_error": tflite_max_abs_error,
        "float32_max_abs_probability_error": float_max_abs_error,
        "dynamic_max_abs_probability_error": dynamic_max_abs_error,
        "float32_colab_inference_ms": float_inference_ms,
        "dynamic_colab_inference_ms": dynamic_inference_ms,
        "benchmark_frames": len(benchmark_input),
    },
    "environment": {
        "run_mode": RUN_MODE,
        "pipeline_revision": PIPELINE_REVISION,
        "tensorflow": tf.__version__,
        "tensorflow_hub": getattr(hub, "__version__", "unknown"),
        "python": os.sys.version,
    },
}
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(json_ready(metrics_report), ensure_ascii=False, indent=2), encoding="utf-8"
)

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(dev_fit_history["loss"], label="개발 데이터 학습")
axes[0].set_title("선택 모델의 Test 이전 학습 손실")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(final_history["loss"], label="전체 데이터 배포 모델 학습", color="#ED7D31")
axes[1].set_title("배포 모델 학습 손실")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()
save_report_figure(fig, "1_learning_curves.png")
plt.show()
plt.close(fig)

# 미사용 그룹 Test confusion matrix
cm = confusion_matrix(test_y, test_predictions, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CATEGORIES,
    yticklabels=CATEGORIES,
    ax=ax,
)
ax.set_title("원본 그룹 분리 Test 혼동행렬")
ax.set_xlabel("예측 클래스")
ax.set_ylabel("실제 클래스")
ax.tick_params(axis="x", labelrotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
save_report_figure(fig, "2_test_confusion_matrix.png")
plt.show()
plt.close(fig)

# 클래스별 Test 지표
precision, recall, class_f1, support = precision_recall_fscore_support(
    test_y, test_predictions, labels=range(NUM_CLASSES), zero_division=0
)
per_class = pd.DataFrame({
    "class": CATEGORIES,
    "precision": precision,
    "recall": recall,
    "f1": class_f1,
    "support": support,
})
per_class.to_csv(OUTPUT_DIR / "test_per_class_metrics.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(15, 6))
per_class.set_index("class")[["precision", "recall", "f1"]].plot.bar(ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("원본 그룹 분리 Test 클래스별 성능")
ax.set_xlabel("클래스")
ax.set_ylabel("점수")
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", labelrotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
save_report_figure(fig, "3_test_per_class_metrics.png")
plt.show()
plt.close(fig)

# Development OOF threshold trade-off
dev_y = champion_summary["y_true"]
dev_probabilities = champion_summary["probabilities"]
threshold_curve = []
for threshold in np.linspace(0.0, 1.0, 101):
    metrics = operating_metrics(dev_y, dev_probabilities, threshold)
    threshold_curve.append({"threshold": threshold, **metrics})
threshold_curve = pd.DataFrame(threshold_curve)
threshold_curve.to_csv(OUTPUT_DIR / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_curve["threshold"], threshold_curve["target_macro_f1"], label="표적 Macro-F1")
ax.plot(threshold_curve["threshold"], threshold_curve["false_alert_rate"], label="비표적 오알림률")
ax.axhline(MAX_FALSE_ALERT_RATE, color="red", linestyle="--", label="허용 오알림률")
ax.set_title("Development OOF 전역 임계값 변화")
ax.set_xlabel("전역 임계값")
ax.set_ylabel("점수 / 비율")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
save_report_figure(fig, "4_threshold_tradeoff.png")
plt.show()
plt.close(fig)

print("\n=== 완료 ===")
print(f"기준선 development CV macro-F1: {baseline['summary']['metrics']['cv_target_macro_f1_mean']:.4f}")
print(f"선택 모델 development CV macro-F1: {champion_summary['metrics']['cv_target_macro_f1_mean']:.4f}")
print(f"Untouched test macro-F1: {test_metrics['target_macro_f1']:.4f}")
print(f"Untouched test false-alert rate: {test_metrics['false_alert_rate']:.4f}")
print(f"Untouched test 비표적음→노크 오알림률: {test_metrics['unknown_to_knock_false_alert_rate']:.4f}")
print(f"Untouched test 비표적음→사이렌 오알림률: {test_metrics['unknown_to_siren_false_alert_rate']:.4f}")
print(f"TFLite 선택: {'dynamic_range' if use_dynamic else 'float32'} ({len(selected_tflite)/1024:.1f} KiB)")
print("평가 산출물:", OUTPUT_DIR)
print("모델 산출물:", MODEL_DIR)
print(f"현재까지 총 실행시간: {(time.perf_counter() - PIPELINE_STARTED) / 60:.1f}분")


## 11. 논문용 효과 분석과 통계 산출물

모델·가중치·임계값은 development OOF에서만 선택하고, 아래 비교는 고정된 설정을 untouched test에 적용합니다.  
핵심 비교는 (1) 기준선 대 하드 네거티브 학습, (2) 전역 임계값 대 클래스별 임계값입니다.


In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


def candidate_test_records(
    candidate: dict[str, Any],
    training_seed: int,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """모든 방법을 전달받은 동일 seed로 학습합니다. Test로 seed를 선택하지 않습니다."""
    config = candidate["family_result"]["config"]
    summary = candidate["summary"]
    if training_seed == PRIMARY_FINAL_TRAINING_SEED and config == champion_config:
        return test_records, summary
    if training_seed == PRIMARY_FINAL_TRAINING_SEED and config == baseline["family_result"]["config"]:
        return baseline_test_records, summary

    activate_hard_negative_selection(candidate.get("hard_negative_paths", ()))
    epochs = max(1, int(round(np.median(candidate["family_result"]["best_epochs"]))))
    model, _ = train_fixed_epochs(dev_df, config, epochs, int(training_seed))
    records = predict_frame_records(
        model, test_df, config.representation, fold_number=-int(training_seed)
    )
    del model
    tf.keras.backend.clear_session()
    return records, summary


def arrays_for_candidate(
    candidate: dict[str, Any],
    training_seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float | np.ndarray]:
    records, summary = candidate_test_records(candidate, training_seed)
    y_true, probabilities, groups = records_to_arrays(
        records,
        summary["pooling"],
        summary["context_frames"],
        summary["temperature"],
    )
    return y_true, probabilities, groups, summary["thresholds"]


# 대표 split의 세 초기화 각각에서 네 방법이 동일 seed를 공유합니다.
arm_test_outputs_by_seed: dict[int, dict[str, dict[str, Any]]] = {}
for training_seed in representative_final_training_seeds:
    current_outputs: dict[str, dict[str, Any]] = {}
    for arm_id, candidate in v3_candidates.items():
        current_y, current_probabilities, current_groups, current_thresholds = arrays_for_candidate(
            candidate, int(training_seed)
        )
        if not np.array_equal(test_y, current_y) or not np.array_equal(test_groups, current_groups):
            raise AssertionError(
                f"seed={training_seed}, {arm_id}: test 표본 순서 또는 그룹이 일치하지 않습니다."
            )
        current_outputs[arm_id] = {
            "y_true": current_y,
            "probabilities": current_probabilities,
            "groups": current_groups,
            "thresholds": current_thresholds,
            "final_training_seed": int(training_seed),
            "candidate": candidate,
        }
    used_seeds = {
        int(output["final_training_seed"]) for output in current_outputs.values()
    }
    if used_seeds != {int(training_seed)}:
        raise AssertionError("같은 대표 split 안에서 방법별 최종 학습 seed가 달라졌습니다.")
    arm_test_outputs_by_seed[int(training_seed)] = current_outputs
activate_hard_negative_selection(champion.get("hard_negative_paths", ()))

# 기존 시각화·TFLite parity에는 사전 지정된 첫 seed만 사용합니다(성능으로 고르지 않음).
arm_test_outputs = arm_test_outputs_by_seed[PRIMARY_FINAL_TRAINING_SEED]
baseline_probabilities = arm_test_outputs["baseline"]["probabilities"]
baseline_thresholds = arm_test_outputs["baseline"]["thresholds"]
hn_probabilities = arm_test_outputs["quota60_2x"]["probabilities"]
hn_thresholds = arm_test_outputs["quota60_2x"]["thresholds"]

# 임계값은 동일한 champion development OOF에서만 결정됩니다.
champion_records = champion["family_result"]["records"]
global_threshold_summary = summarize_records(
    champion_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    champion_summary["temperature"],
    threshold_mode="global",
)
class_threshold_summary = summarize_records(
    champion_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    champion_summary["temperature"],
    threshold_mode="per_class",
)


def expanded_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    thresholds: float | np.ndarray,
) -> dict[str, float]:
    metrics = operating_metrics(y_true, probabilities, thresholds)
    prediction = decision_predictions(probabilities, thresholds)
    one_hot = np.eye(NUM_CLASSES, dtype=np.float64)[y_true]
    target_mask = y_true != UNKNOWN_ID
    unknown_mask = ~target_mask
    metrics.update({
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "matthews_correlation_coefficient": float(matthews_corrcoef(y_true, prediction)),
        "negative_log_likelihood": float(log_loss(y_true, probabilities, labels=range(NUM_CLASSES))),
        "multiclass_brier_score": float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))),
        "target_correct_class_rate": float(np.mean(prediction[target_mask] == y_true[target_mask])),
        "target_rejection_rate": float(np.mean(prediction[target_mask] == UNKNOWN_ID)),
        "unknown_specificity": float(np.mean(prediction[unknown_mask] == UNKNOWN_ID)),
    })
    return metrics


NUMERIC_PAPER_METRICS = [
    "target_macro_f1", "all_class_macro_f1", "accuracy", "false_alert_rate",
    "unknown_to_knock_false_alert_rate", "unknown_to_siren_false_alert_rate",
    "ece", "balanced_accuracy", "matthews_correlation_coefficient",
    "negative_log_likelihood", "multiclass_brier_score",
    "target_correct_class_rate", "target_rejection_rate", "unknown_specificity",
]

# 4방법 × 3초기화의 원시 결과: 논문 분석의 기본 단위입니다.
initialization_rows: list[dict[str, Any]] = []
for training_seed, outputs in arm_test_outputs_by_seed.items():
    for arm_id, output in outputs.items():
        candidate = output["candidate"]
        config = candidate["family_result"]["config"]
        initialization_rows.append({
            "outer_split_seed": int(SEED),
            "final_training_seed": int(training_seed),
            "method": arm_id,
            "head": config.head,
            "representation": config.representation,
            "hard_negative_weight": config.hard_negative_weight,
            "selection_policy": candidate["selection_policy"],
            "selected_negative_files": len(candidate["hard_negative_paths"]),
            "selected_true_hard_negatives": candidate["selected_true_hard_negatives"],
            "selected_semihard_negatives": candidate["selected_semihard_negatives"],
            **expanded_metrics(test_y, output["probabilities"], output["thresholds"]),
        })
paper_main_results_by_initialization = pd.DataFrame(initialization_rows)
expected_pairs = {
    (int(seed), arm_id)
    for seed in representative_final_training_seeds
    for arm_id in V3_ARM_SPECS
}
actual_pairs = set(zip(
    paper_main_results_by_initialization["final_training_seed"].astype(int),
    paper_main_results_by_initialization["method"].astype(str),
))
if actual_pairs != expected_pairs:
    raise AssertionError("대표 split의 방법×초기화 조합이 4×3을 충족하지 못했습니다.")
paper_main_results_by_initialization.to_csv(
    PAPER_DIR / "paper_main_results_by_initialization.csv", index=False, encoding="utf-8-sig"
)

summary = paper_main_results_by_initialization.groupby("method")[NUMERIC_PAPER_METRICS].agg(
    ["mean", "std", "min", "max"]
)
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
method_metadata = pd.DataFrame([
    {
        "method": arm_id,
        "display_name": candidate["display_name"],
        "selection_policy": candidate["selection_policy"],
        "hard_negative_weight": candidate["family_result"]["config"].hard_negative_weight,
        "selected_negative_files": len(candidate["hard_negative_paths"]),
        "selected_true_hard_negatives": candidate["selected_true_hard_negatives"],
        "selected_semihard_negatives": candidate["selected_semihard_negatives"],
        "initialization_repetitions": len(representative_final_training_seeds),
    }
    for arm_id, candidate in v3_candidates.items()
])
paper_main_results = method_metadata.merge(summary, on="method", validate="one_to_one")
paper_main_results.to_csv(
    PAPER_DIR / "paper_main_results.csv", index=False, encoding="utf-8-sig"
)

# Paired initialization effect sizes: method A/B always have the same final_training_seed.
effect_rows: list[dict[str, Any]] = []
for training_seed in representative_final_training_seeds:
    current = paper_main_results_by_initialization[
        paper_main_results_by_initialization["final_training_seed"] == int(training_seed)
    ].set_index("method")
    for arm_id in ["raw_top1_2x", "raw_plus_semihard_p010_1p5x", "quota60_2x"]:
        a = current.loc["baseline"]
        b = current.loc[arm_id]
        effect_rows.append({
            "comparison": f"baseline_vs_{arm_id}",
            "final_training_seed": int(training_seed),
            "target_macro_f1_delta_b_minus_a": float(b["target_macro_f1"] - a["target_macro_f1"]),
            "accuracy_delta_b_minus_a": float(b["accuracy"] - a["accuracy"]),
            "false_alert_absolute_reduction_a_minus_b": float(a["false_alert_rate"] - b["false_alert_rate"]),
            "unknown_to_knock_absolute_reduction": float(
                a["unknown_to_knock_false_alert_rate"] - b["unknown_to_knock_false_alert_rate"]
            ),
        })
paper_effect_sizes_by_initialization = pd.DataFrame(effect_rows)
paper_effect_sizes_by_initialization.to_csv(
    PAPER_DIR / "paper_effect_sizes_by_initialization.csv", index=False, encoding="utf-8-sig"
)
effect_metrics = [
    "target_macro_f1_delta_b_minus_a", "accuracy_delta_b_minus_a",
    "false_alert_absolute_reduction_a_minus_b", "unknown_to_knock_absolute_reduction",
]
paper_effect_sizes = paper_effect_sizes_by_initialization.groupby("comparison")[effect_metrics].agg(
    ["mean", "std", "min", "max"]
)
paper_effect_sizes.columns = [f"{metric}_{stat}" for metric, stat in paper_effect_sizes.columns]
paper_effect_sizes = paper_effect_sizes.reset_index()
paper_effect_sizes.to_csv(
    PAPER_DIR / "paper_effect_sizes.csv", index=False, encoding="utf-8-sig"
)

# 첫 번째 사전 지정 seed의 상세 클래스/환경 분석. seed 선택 근거는 성능이 아닙니다.
methods: dict[str, dict[str, Any]] = {}
for arm_id, output in arm_test_outputs.items():
    candidate = output["candidate"]
    methods[arm_id] = {
        "probabilities": output["probabilities"],
        "thresholds": output["thresholds"],
        "config": candidate["family_result"]["config"],
        "selection_policy": candidate["selection_policy"],
        "selected_negative_files": len(candidate["hard_negative_paths"]),
        "selected_true_hard_negatives": candidate["selected_true_hard_negatives"],
        "selected_semihard_negatives": candidate["selected_semihard_negatives"],
    }
methods["champion_global_threshold"] = {
    "probabilities": test_probabilities,
    "thresholds": global_threshold_summary["thresholds"],
    "config": champion_config,
    "selection_policy": champion.get("selection_policy", "none"),
    "selected_negative_files": len(champion.get("hard_negative_paths", ())),
    "selected_true_hard_negatives": champion.get("selected_true_hard_negatives", 0),
    "selected_semihard_negatives": champion.get("selected_semihard_negatives", 0),
}
methods["champion_class_thresholds"] = {
    **methods["champion_global_threshold"],
    "thresholds": class_threshold_summary["thresholds"],
}
reference_rows = []
for method_name, method in methods.items():
    reference_rows.append({
        "method": method_name,
        "outer_split_seed": int(SEED),
        "final_training_seed": int(PRIMARY_FINAL_TRAINING_SEED),
        "hard_negative_weight": method["config"].hard_negative_weight,
        "selection_policy": method["selection_policy"],
        "selected_negative_files": method["selected_negative_files"],
        **expanded_metrics(test_y, method["probabilities"], method["thresholds"]),
    })
paper_reference_seed_results = pd.DataFrame(reference_rows)
paper_reference_seed_results.to_csv(
    PAPER_DIR / "paper_reference_seed_results.csv", index=False, encoding="utf-8-sig"
)


def threshold_vector(thresholds: float | np.ndarray) -> np.ndarray:
    values = (
        np.full(NUM_CLASSES, float(thresholds), dtype=np.float64)
        if np.isscalar(thresholds)
        else np.asarray(thresholds, dtype=np.float64).copy()
    )
    values[UNKNOWN_ID] = 0.0
    return values


def per_class_rows(
    method_name: str,
    probabilities: np.ndarray,
    thresholds: float | np.ndarray,
) -> list[dict[str, Any]]:
    prediction = decision_predictions(probabilities, thresholds)
    matrix = confusion_matrix(test_y, prediction, labels=range(NUM_CLASSES))
    values = threshold_vector(thresholds)
    rows = []
    for class_id, label in enumerate(CATEGORIES):
        tp = int(matrix[class_id, class_id])
        fn = int(matrix[class_id, :].sum() - tp)
        fp = int(matrix[:, class_id].sum() - tp)
        tn = int(matrix.sum() - tp - fn - fp)
        binary_true = (test_y == class_id).astype(np.int64)
        score = probabilities[:, class_id]
        rows.append({
            "method": method_name,
            "class_id": class_id,
            "class": label,
            "threshold": float(values[class_id]),
            "support": int(tp + fn),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": float(tp / (tp + fp)) if tp + fp else 0.0,
            "recall_sensitivity": float(tp / (tp + fn)) if tp + fn else 0.0,
            "specificity": float(tn / (tn + fp)) if tn + fp else float("nan"),
            "fpr": float(fp / (fp + tn)) if fp + tn else float("nan"),
            "fnr": float(fn / (fn + tp)) if fn + tp else float("nan"),
            "f1": float(2 * tp / (2 * tp + fp + fn)) if 2 * tp + fp + fn else 0.0,
            "roc_auc_ovr": float(roc_auc_score(binary_true, score)) if len(np.unique(binary_true)) == 2 else float("nan"),
            "average_precision": float(average_precision_score(binary_true, score)) if binary_true.any() else float("nan"),
            "unknown_to_class_false_alert_rate": (
                float(np.mean(prediction[test_y == UNKNOWN_ID] == class_id))
                if class_id != UNKNOWN_ID and np.any(test_y == UNKNOWN_ID)
                else 0.0
            ),
        })
    return rows


paper_per_class_metrics = pd.DataFrame([
    row
    for method_name, method in methods.items()
    for row in per_class_rows(method_name, method["probabilities"], method["thresholds"])
])
paper_per_class_metrics.to_csv(
    PAPER_DIR / "paper_per_class_metrics.csv", index=False, encoding="utf-8-sig"
)


def environment_false_alert_rows(
    method_name: str,
    probabilities: np.ndarray,
    thresholds: float | np.ndarray,
    split_seed: int = SEED,
) -> list[dict[str, Any]]:
    """공개 생활소음 범주별 오탐률을 계산합니다."""
    prediction = decision_predictions(probabilities, thresholds)
    ordered_test = test_df.sort_values("relative_path").reset_index(drop=True)
    environment = ordered_test["environment_category"].astype(str).to_numpy()
    rows = []
    for category in OPEN_HARD_NEGATIVE_QUERIES:
        mask = (test_y == UNKNOWN_ID) & (environment == category)
        support = int(mask.sum())
        rows.append({
            "split_seed": int(split_seed),
            "method": method_name,
            "environment_category": category,
            "support": support,
            "false_alert_rate": float(np.mean(prediction[mask] != UNKNOWN_ID)) if support else float("nan"),
            "unknown_to_knock_false_alert_rate": float(np.mean(np.isin(prediction[mask], KNOCK_IDS))) if support else float("nan"),
            "unknown_to_siren_false_alert_rate": float(np.mean(np.isin(prediction[mask], SIREN_IDS))) if support else float("nan"),
        })
    return rows


paper_environment_false_alerts = pd.DataFrame([
    row
    for method_name, method in methods.items()
    for row in environment_false_alert_rows(
        method_name, method["probabilities"], method["thresholds"]
    )
])
paper_environment_false_alerts.to_csv(
    PAPER_DIR / "paper_environment_false_alerts.csv", index=False, encoding="utf-8-sig"
)



def grouped_paired_bootstrap(
    name: str,
    probabilities_a: np.ndarray,
    thresholds_a: float | np.ndarray,
    probabilities_b: np.ndarray,
    thresholds_b: float | np.ndarray,
    iterations: int = BOOTSTRAP_ITERATIONS,
) -> dict[str, Any]:
    unique_groups = np.unique(test_groups)
    indices_by_group = {group: np.flatnonzero(test_groups == group) for group in unique_groups}
    rng = np.random.default_rng(SEED + stable_seed("paper-bootstrap", name))
    deltas = {"target_macro_f1_delta_b_minus_a": [], "false_alert_reduction_a_minus_b": [], "accuracy_delta_b_minus_a": []}
    for _ in range(iterations):
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([indices_by_group[group] for group in sampled])
        a = operating_metrics(test_y[indices], probabilities_a[indices], thresholds_a)
        b = operating_metrics(test_y[indices], probabilities_b[indices], thresholds_b)
        deltas["target_macro_f1_delta_b_minus_a"].append(b["target_macro_f1"] - a["target_macro_f1"])
        if np.isfinite(a["false_alert_rate"]) and np.isfinite(b["false_alert_rate"]):
            deltas["false_alert_reduction_a_minus_b"].append(a["false_alert_rate"] - b["false_alert_rate"])
        deltas["accuracy_delta_b_minus_a"].append(b["accuracy"] - a["accuracy"])

    result: dict[str, Any] = {"comparison": name, "bootstrap_iterations": iterations}
    for metric_name, series in deltas.items():
        values = np.asarray(series, dtype=np.float64)
        result[f"{metric_name}_mean"] = float(values.mean())
        result[f"{metric_name}_ci95_low"] = float(np.percentile(values, 2.5))
        result[f"{metric_name}_ci95_high"] = float(np.percentile(values, 97.5))
        result[f"{metric_name}_probability_positive"] = float(np.mean(values > 0.0))
    return result


def exact_paired_test(
    name: str,
    probabilities_a: np.ndarray,
    thresholds_a: float | np.ndarray,
    probabilities_b: np.ndarray,
    thresholds_b: float | np.ndarray,
) -> dict[str, Any]:
    prediction_a = decision_predictions(probabilities_a, thresholds_a)
    prediction_b = decision_predictions(probabilities_b, thresholds_b)
    correct_a = prediction_a == test_y
    correct_b = prediction_b == test_y
    a_wrong_b_right = int(np.sum(~correct_a & correct_b))
    a_right_b_wrong = int(np.sum(correct_a & ~correct_b))
    discordant = a_wrong_b_right + a_right_b_wrong
    p_value = float(
        binomtest(a_wrong_b_right, discordant, 0.5, alternative="two-sided").pvalue
    ) if discordant else 1.0

    unknown = test_y == UNKNOWN_ID
    false_a = prediction_a[unknown] != UNKNOWN_ID
    false_b = prediction_b[unknown] != UNKNOWN_ID
    a_false_b_correct = int(np.sum(false_a & ~false_b))
    a_correct_b_false = int(np.sum(~false_a & false_b))
    false_discordant = a_false_b_correct + a_correct_b_false
    false_p = float(
        binomtest(a_false_b_correct, false_discordant, 0.5, alternative="two-sided").pvalue
    ) if false_discordant else 1.0
    return {
        "comparison": name,
        "analysis_level": "file-level exploratory exact paired test",
        "accuracy_a_wrong_b_right": a_wrong_b_right,
        "accuracy_a_right_b_wrong": a_right_b_wrong,
        "accuracy_exact_mcnemar_p": p_value,
        "unknown_a_false_b_correct": a_false_b_correct,
        "unknown_a_correct_b_false": a_correct_b_false,
        "unknown_false_alert_exact_mcnemar_p": false_p,
    }


comparison_inputs: list[dict[str, Any]] = []
for training_seed in representative_final_training_seeds:
    outputs = arm_test_outputs_by_seed[int(training_seed)]
    baseline_output = outputs["baseline"]
    for arm_id in ["raw_top1_2x", "raw_plus_semihard_p010_1p5x", "quota60_2x"]:
        comparison_inputs.append({
            "comparison": f"baseline_vs_{arm_id}",
            "final_training_seed": int(training_seed),
            "probabilities_a": baseline_output["probabilities"],
            "thresholds_a": baseline_output["thresholds"],
            "probabilities_b": outputs[arm_id]["probabilities"],
            "thresholds_b": outputs[arm_id]["thresholds"],
        })
    champion_probabilities_for_seed = outputs[selected_arm_id]["probabilities"]
    comparison_inputs.append({
        "comparison": "global_vs_class_thresholds",
        "final_training_seed": int(training_seed),
        "probabilities_a": champion_probabilities_for_seed,
        "thresholds_a": global_threshold_summary["thresholds"],
        "probabilities_b": champion_probabilities_for_seed,
        "thresholds_b": class_threshold_summary["thresholds"],
    })

bootstrap_rows = []
exact_rows = []
for item in comparison_inputs:
    name_with_seed = f"{item['comparison']}__seed_{item['final_training_seed']}"
    bootstrap_rows.append({
        "comparison_family": item["comparison"],
        "final_training_seed": item["final_training_seed"],
        **grouped_paired_bootstrap(
            name_with_seed,
            item["probabilities_a"], item["thresholds_a"],
            item["probabilities_b"], item["thresholds_b"],
        ),
    })
    exact_rows.append({
        "comparison_family": item["comparison"],
        "final_training_seed": item["final_training_seed"],
        **exact_paired_test(
            name_with_seed,
            item["probabilities_a"], item["thresholds_a"],
            item["probabilities_b"], item["thresholds_b"],
        ),
    })
paper_bootstrap = pd.DataFrame(bootstrap_rows)
paper_exact_tests = pd.DataFrame(exact_rows)
paper_bootstrap.to_csv(PAPER_DIR / "paper_grouped_bootstrap.csv", index=False, encoding="utf-8-sig")
paper_exact_tests.to_csv(PAPER_DIR / "paper_exact_paired_tests.csv", index=False, encoding="utf-8-sig")

bootstrap_summary_metrics = [
    "target_macro_f1_delta_b_minus_a_mean",
    "false_alert_reduction_a_minus_b_mean",
    "accuracy_delta_b_minus_a_mean",
]
paper_bootstrap_initialization_summary = paper_bootstrap.groupby("comparison_family")[
    bootstrap_summary_metrics
].agg(["mean", "std", "min", "max"])
paper_bootstrap_initialization_summary.columns = [
    f"{metric}_{stat}" for metric, stat in paper_bootstrap_initialization_summary.columns
]
paper_bootstrap_initialization_summary = paper_bootstrap_initialization_summary.reset_index()
paper_bootstrap_initialization_summary.to_csv(
    PAPER_DIR / "paper_grouped_bootstrap_initialization_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

threshold_ablation_rows = []
for mode_name, summary in [
    ("global", global_threshold_summary),
    ("per_class", class_threshold_summary),
]:
    threshold_ablation_rows.append({
        "threshold_mode": mode_name,
        "development_oof_thresholds": json.dumps(
            json_ready(summary["thresholds"]), ensure_ascii=False
        ),
        **{f"development_oof_{key}": value for key, value in summary["metrics"].items()},
        **{f"untouched_test_{key}": value for key, value in expanded_metrics(
            test_y, test_probabilities, summary["thresholds"]
        ).items()},
    })
paper_threshold_ablation = pd.DataFrame(threshold_ablation_rows)
paper_threshold_ablation.to_csv(
    PAPER_DIR / "paper_threshold_ablation.csv", index=False, encoding="utf-8-sig"
)

paper_siren_label_dictionary = pd.DataFrame([
    {
        "class_label": label,
        "international_term": definition["international_term"],
        "operational_definition": definition["operational_definition"],
        "legacy_source_labels": "|".join(sorted(
            source_label for source_label, canonical_label in LEGACY_RELABEL.items()
            if canonical_label == label
        )),
        "source_path_policy": "원본 relative_path 유지; label 열만 표준화",
    }
    for label, definition in SIREN_LABEL_DEFINITIONS.items()
])
paper_siren_label_dictionary.to_csv(
    PAPER_DIR / "paper_siren_label_dictionary.csv", index=False, encoding="utf-8-sig"
)

paper_dataset_summary = (
    metadata.groupby("label", sort=False)
    .agg(
        files=("relative_path", "size"),
        groups=("group_id", "nunique"),
        duration_seconds=("duration_seconds", "sum"),
        duration_mean_seconds=("duration_seconds", "mean"),
    )
    .reindex(CATEGORIES)
    .reset_index()
)
paper_dataset_summary["development_files"] = (
    dev_df.groupby("label").size().reindex(CATEGORIES, fill_value=0).to_numpy()
)
paper_dataset_summary["test_files"] = (
    test_df.groupby("label").size().reindex(CATEGORIES, fill_value=0).to_numpy()
)
paper_dataset_summary.to_csv(
    PAPER_DIR / "paper_dataset_summary.csv", index=False, encoding="utf-8-sig"
)

# LaTeX 표는 UTF-8로 저장하며 논문에서 필요한 열만 포함합니다.
latex_main = paper_main_results[[
    "method", "target_macro_f1_mean", "target_macro_f1_std",
    "accuracy_mean", "false_alert_rate_mean", "false_alert_rate_std",
    "unknown_to_knock_false_alert_rate_mean", "unknown_to_siren_false_alert_rate_mean",
]].to_latex(index=False, float_format=lambda value: f"{value:.4f}", escape=True)
(PAPER_DIR / "table_main_results.tex").write_text(latex_main, encoding="utf-8")

latex_thresholds = paper_per_class_metrics[
    paper_per_class_metrics["method"] == "champion_class_thresholds"
][["class", "threshold", "precision", "recall_sensitivity", "specificity", "f1"]].to_latex(
    index=False, float_format=lambda value: f"{value:.4f}", escape=True
)
(PAPER_DIR / "table_class_thresholds.tex").write_text(latex_thresholds, encoding="utf-8")

# Figure 5: 동일한 세 초기화 seed에서 계산한 네 방법 평균±표준편차
arm_order = list(V3_ARM_SPECS)
summary_by_method = paper_main_results.set_index("method").loc[arm_order]
plot_values = summary_by_method[[
    "target_macro_f1_mean", "false_alert_rate_mean", "unknown_to_knock_false_alert_rate_mean"
]].copy()
plot_errors = summary_by_method[[
    "target_macro_f1_std", "false_alert_rate_std", "unknown_to_knock_false_alert_rate_std"
]].copy()
plot_values.index = [V3_ARM_SPECS[arm_id]["display_name"] for arm_id in arm_order]
plot_errors.index = plot_values.index
plot_values.columns = ["표적 Macro-F1", "생활소음 오탐률", "생활소음→노크 오탐률"]
plot_errors.columns = plot_values.columns
fig, ax = plt.subplots(figsize=(15, 7))
plot_values.plot.bar(
    ax=ax,
    yerr=plot_errors,
    capsize=4,
    color=["#4472C4", "#ED7D31", "#70AD47"],
)
ax.set_ylim(0, 1.05)
ax.set_title("동일 초기화 3회 반복: 네 하드 네거티브 방법 평균±표준편차")
ax.set_ylabel("비율")
ax.set_xlabel("")
ax.tick_params(axis="x", labelrotation=18)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.grid(axis="y", alpha=0.3)
save_report_figure(fig, "5_paper_hard_negative_comparison.png")
plt.show()
plt.close(fig)

# Figure 6: OOF에서 결정된 전역/클래스별 임계값
global_values = threshold_vector(global_threshold_summary["thresholds"])
class_values = threshold_vector(class_threshold_summary["thresholds"])
threshold_frame = pd.DataFrame({
    "클래스": CATEGORIES[:UNKNOWN_ID],
    "전역 임계값": global_values[:UNKNOWN_ID],
    "클래스별 임계값": class_values[:UNKNOWN_ID],
}).set_index("클래스")
fig, ax = plt.subplots(figsize=(14, 6))
threshold_frame.plot.bar(ax=ax, color=["#A5A5A5", "#5B9BD5"])
ax.set_ylim(0, 1.05)
ax.set_title("Development OOF에서 결정된 클래스별 임계값")
ax.set_ylabel("임계값")
ax.set_xlabel("클래스")
ax.tick_params(axis="x", labelrotation=35)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.grid(axis="y", alpha=0.3)
save_report_figure(fig, "6_paper_class_thresholds.png")
plt.show()
plt.close(fig)

# Figure 7/8: champion 확률의 클래스별 ROC 및 PR 곡선
fig_roc, ax_roc = plt.subplots(figsize=(9, 8))
fig_pr, ax_pr = plt.subplots(figsize=(9, 8))
for class_id, label in enumerate(CATEGORIES[:UNKNOWN_ID]):
    binary_true = (test_y == class_id).astype(np.int64)
    if len(np.unique(binary_true)) < 2:
        continue
    score = test_probabilities[:, class_id]
    fpr, tpr, _ = roc_curve(binary_true, score)
    precision_curve, recall_curve, _ = precision_recall_curve(binary_true, score)
    roc_value = roc_auc_score(binary_true, score)
    ap_value = average_precision_score(binary_true, score)
    ax_roc.plot(fpr, tpr, label=f"{label} (AUC={roc_value:.3f})")
    ax_pr.plot(recall_curve, precision_curve, label=f"{label} (AP={ap_value:.3f})")
ax_roc.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax_roc.set(title="Untouched test 클래스별 ROC", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax_roc.grid(alpha=0.3)
ax_roc.legend(fontsize=8)
save_report_figure(fig_roc, "7_paper_per_class_roc.png")
plt.show()
plt.close(fig_roc)
ax_pr.set(title="Untouched test 클래스별 Precision–Recall", xlabel="Recall", ylabel="Precision")
ax_pr.grid(alpha=0.3)
ax_pr.legend(fontsize=8)
save_report_figure(fig_pr, "8_paper_per_class_pr.png")
plt.show()
plt.close(fig_pr)

# Figure 9: 신뢰도 보정 상태(reliability diagram)
confidence = test_probabilities.max(axis=1)
raw_prediction = test_probabilities.argmax(axis=1)
correct = (raw_prediction == test_y).astype(float)
calibration_rows = []
for lower, upper in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):
    mask = (confidence > lower) & (confidence <= upper)
    if mask.any():
        calibration_rows.append({
            "confidence": float(confidence[mask].mean()),
            "accuracy": float(correct[mask].mean()),
            "count": int(mask.sum()),
        })
paper_calibration = pd.DataFrame(calibration_rows)
paper_calibration.to_csv(PAPER_DIR / "paper_calibration_bins.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "--", color="gray", label="완전 보정")
ax.plot(paper_calibration["confidence"], paper_calibration["accuracy"], "o-", label="Champion")
ax.set(title="Untouched test 신뢰도 보정", xlabel="평균 신뢰도", ylabel="실제 정확도", xlim=(0, 1), ylim=(0, 1))
ax.grid(alpha=0.3)
ax.legend()
save_report_figure(fig, "9_paper_reliability_diagram.png")
plt.show()
plt.close(fig)



def run_repeated_group_evaluation() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """5 outer splits × 3 matched initialization seeds × 4 methods."""
    global dev_df, test_df, inner_splits, inner_splits_count, inner_seed, ACTIVE_TRAINING_SEED
    original_state = (
        dev_df, test_df, inner_splits, inner_splits_count, inner_seed, ACTIVE_TRAINING_SEED
    )
    result_rows: list[dict[str, Any]] = []
    mining_rows: list[pd.DataFrame] = []
    environment_rows: list[dict[str, Any]] = []
    repeat_signature = hashlib.sha256(json.dumps({
        "pipeline_revision": PIPELINE_REVISION,
        "oof_cache_revision": OOF_CACHE_REVISION,
        "final_training_seed_offsets": FINAL_TRAINING_SEED_OFFSETS,
        "arms": V3_ARM_SPECS,
        "manifest": metadata[["relative_path", "label", "group_id", "audio_hash"]]
            .sort_values("relative_path").to_dict("records"),
        "inner_folds": INNER_FOLDS,
        "max_epochs": MAX_EPOCHS,
    }, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    repeat_cache_dir = CHECKPOINT_ROOT / "v3_2_matched_seed_cache"
    repeat_cache_dir.mkdir(parents=True, exist_ok=True)
    try:
        for split_seed in REPEATED_SPLIT_SEEDS:
            matched_seeds = final_training_seeds(int(split_seed))
            print(f"\n[반복 grouped v3.2] split={split_seed}, final seeds={matched_seeds}")
            repeat_cache = repeat_cache_dir / f"split_{int(split_seed)}_{repeat_signature}.pkl"
            if repeat_cache.exists():
                with repeat_cache.open("rb") as file:
                    cached_split = pickle.load(file)
                if (
                    cached_split.get("signature") == repeat_signature
                    and cached_split.get("split_seed") == int(split_seed)
                    and tuple(cached_split.get("final_training_seeds", ())) == matched_seeds
                ):
                    result_rows.extend(cached_split["result_rows"])
                    mining_rows.extend(cached_split["mining_rows"])
                    environment_rows.extend(cached_split["environment_rows"])
                    print("  split 최종 평가 캐시 재사용:", repeat_cache.name)
                    continue
            result_start = len(result_rows)
            mining_start = len(mining_rows)
            environment_start = len(environment_rows)

            train_idx, current_test_idx, split_score = select_outer_holdout(
                metadata, seed=int(split_seed)
            )
            dev_df = metadata.iloc[train_idx].copy().reset_index(drop=True)
            test_df = metadata.iloc[current_test_idx].copy().reset_index(drop=True)
            activate_hard_negative_selection(())
            if not set(dev_df["group_id"]).isdisjoint(set(test_df["group_id"])):
                raise AssertionError(f"split={split_seed}: outer group leakage")
            inner_splits_count = feasible_splits(dev_df, INNER_FOLDS, f"repeat-{split_seed}")
            inner_splits, _, inner_seed = select_inner_splits(
                dev_df, inner_splits_count, seed=int(split_seed) + 1
            )
            ACTIVE_TRAINING_SEED = int(split_seed)

            baseline_config = FamilyConfig(
                name=f"repeat_s{split_seed}_v3_baseline",
                head="baseline",
                representation="clip_mean",
            )
            baseline_family_current = run_cv_family(baseline_config)
            baseline_summary_current = baseline_family_current["summaries"]["mean_probability"]
            _scored, selections = build_v3_hard_negative_selections(
                baseline_family_current["records"], "mean_probability"
            )

            arm_candidates: dict[str, dict[str, Any]] = {}
            for arm_id, spec in V3_ARM_SPECS.items():
                selected = selections[arm_id]
                selected_paths = tuple(sorted(selected["relative_path"].astype(str)))
                activate_hard_negative_selection(selected_paths)
                if arm_id == "baseline":
                    config = baseline_config
                    family = baseline_family_current
                    summary = baseline_summary_current
                else:
                    config = FamilyConfig(
                        name=f"repeat_s{split_seed}_v3_{arm_id}",
                        head="baseline",
                        representation="clip_mean",
                        hard_negative_weight=float(spec["hard_negative_weight"]),
                    )
                    family = run_cv_family(config)
                    summary = family["summaries"]["mean_probability"]
                    logged = selected.copy()
                    logged.insert(0, "method", arm_id)
                    logged.insert(0, "split_seed", int(split_seed))
                    mining_rows.append(logged)
                true_count = int(selected.get("is_true_hard_negative", pd.Series(dtype=bool)).sum())
                arm_candidates[arm_id] = {
                    "family_result": family,
                    "summary": summary,
                    "arm_id": arm_id,
                    "selection_policy": spec["selection_policy"],
                    "hard_negative_paths": selected_paths,
                    "selected_true_hard_negatives": true_count,
                    "selected_semihard_negatives": int(len(selected) - true_count),
                }

            # OOF-only selection is fixed before any test fit.
            development_selected_arm = max(
                arm_candidates.items(), key=lambda item: candidate_key(item[1])
            )[0]

            for initialization_index, final_seed in enumerate(matched_seeds, start=1):
                fitted_seed_by_method: dict[str, int] = {}
                for arm_id, candidate in arm_candidates.items():
                    config = candidate["family_result"]["config"]
                    summary = candidate["summary"]
                    activate_hard_negative_selection(candidate["hard_negative_paths"])
                    epochs = max(1, int(round(np.median(candidate["family_result"]["best_epochs"]))))
                    fitted, _ = train_fixed_epochs(dev_df, config, epochs, int(final_seed))
                    fitted_seed_by_method[arm_id] = int(final_seed)
                    records = predict_frame_records(
                        fitted, test_df, config.representation, fold_number=-int(final_seed)
                    )
                    current_y, current_probabilities, current_groups = records_to_arrays(
                        records,
                        summary["pooling"],
                        summary["context_frames"],
                        summary["temperature"],
                    )
                    del fitted
                    tf.keras.backend.clear_session()
                    metrics = expanded_metrics(current_y, current_probabilities, summary["thresholds"])
                    result_rows.append({
                        "split_seed": int(split_seed),
                        "initialization_index": int(initialization_index),
                        "final_training_seed": int(final_seed),
                        "method": arm_id,
                        "selection_policy": candidate["selection_policy"],
                        "hard_negative_weight": config.hard_negative_weight,
                        "development_selected": arm_id == development_selected_arm,
                        "outer_split_score": float(split_score),
                        "development_files": int(len(dev_df)),
                        "test_files": int(len(test_df)),
                        "development_groups": int(dev_df["group_id"].nunique()),
                        "test_groups": int(test_df["group_id"].nunique()),
                        "selected_epochs": int(epochs),
                        "hard_negative_files": int(len(candidate["hard_negative_paths"])),
                        "hard_negative_true_errors": candidate["selected_true_hard_negatives"],
                        "hard_negative_semihard": candidate["selected_semihard_negatives"],
                        "thresholds": json.dumps(json_ready(summary["thresholds"]), ensure_ascii=False),
                        **metrics,
                    })

                    ordered_test = test_df.sort_values("relative_path").reset_index(drop=True)
                    environment = ordered_test["environment_category"].astype(str).to_numpy()
                    current_prediction = decision_predictions(current_probabilities, summary["thresholds"])
                    for category in OPEN_HARD_NEGATIVE_QUERIES:
                        mask = (current_y == UNKNOWN_ID) & (environment == category)
                        support = int(mask.sum())
                        environment_rows.append({
                            "split_seed": int(split_seed),
                            "initialization_index": int(initialization_index),
                            "final_training_seed": int(final_seed),
                            "method": arm_id,
                            "environment_category": category,
                            "support": support,
                            "false_alert_rate": float(np.mean(current_prediction[mask] != UNKNOWN_ID)) if support else float("nan"),
                            "unknown_to_knock_false_alert_rate": float(np.mean(np.isin(current_prediction[mask], KNOCK_IDS))) if support else float("nan"),
                            "unknown_to_siren_false_alert_rate": float(np.mean(np.isin(current_prediction[mask], SIREN_IDS))) if support else float("nan"),
                        })
                if set(fitted_seed_by_method) != set(V3_ARM_SPECS):
                    raise AssertionError("한 초기화에서 네 방법 중 일부가 누락되었습니다.")
                if set(fitted_seed_by_method.values()) != {int(final_seed)}:
                    raise AssertionError("같은 outer split/초기화 안에서 방법별 학습 seed가 다릅니다.")

            split_cache_payload = {
                "signature": repeat_signature,
                "split_seed": int(split_seed),
                "final_training_seeds": matched_seeds,
                "result_rows": result_rows[result_start:],
                "mining_rows": mining_rows[mining_start:],
                "environment_rows": environment_rows[environment_start:],
            }
            temporary = repeat_cache.with_suffix(".tmp")
            with temporary.open("wb") as file:
                pickle.dump(split_cache_payload, file, protocol=pickle.HIGHEST_PROTOCOL)
            temporary.replace(repeat_cache)
            print("  split 최종 평가 캐시 저장:", repeat_cache.name)
    finally:
        (
            dev_df, test_df, inner_splits, inner_splits_count, inner_seed, ACTIVE_TRAINING_SEED
        ) = original_state

    results = pd.DataFrame(result_rows)
    numeric_metrics = [
        "target_macro_f1", "all_class_macro_f1", "accuracy", "false_alert_rate",
        "unknown_to_knock_false_alert_rate", "unknown_to_siren_false_alert_rate",
        "ece", "balanced_accuracy", "matthews_correlation_coefficient",
        "negative_log_likelihood", "multiclass_brier_score",
    ]
    required = {"split_seed", "final_training_seed", "method", *numeric_metrics}
    missing = sorted(required - set(results.columns))
    if missing:
        raise RuntimeError(f"반복 평가 결과 열 누락: {missing}")
    expected_count = len(REPEATED_SPLIT_SEEDS) * len(FINAL_TRAINING_SEED_OFFSETS) * len(V3_ARM_SPECS)
    if len(results) != expected_count:
        raise RuntimeError(f"반복 평가 행 수 오류: {len(results)} != {expected_count}")
    per_cell_counts = results.groupby(["split_seed", "final_training_seed"])["method"].nunique()
    if not (per_cell_counts == len(V3_ARM_SPECS)).all():
        raise AssertionError("split×초기화 셀에 네 방법이 모두 존재하지 않습니다.")

    summary = results.groupby("method")[numeric_metrics].agg(["mean", "std", "min", "max"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary = summary.reset_index()

    split_initialization_summary = results.groupby(["split_seed", "method"])[numeric_metrics].agg(
        ["mean", "std", "min", "max"]
    )
    split_initialization_summary.columns = [
        f"{metric}_{stat}" for metric, stat in split_initialization_summary.columns
    ]
    split_initialization_summary = split_initialization_summary.reset_index()

    delta_rows: list[dict[str, Any]] = []
    for (split_seed, final_seed), current in results.groupby(["split_seed", "final_training_seed"]):
        indexed = current.set_index("method")
        row: dict[str, Any] = {
            "split_seed": int(split_seed),
            "final_training_seed": int(final_seed),
        }
        for arm_id in ["raw_top1_2x", "raw_plus_semihard_p010_1p5x", "quota60_2x"]:
            row[f"target_macro_f1_{arm_id}_minus_baseline"] = float(
                indexed.loc[arm_id, "target_macro_f1"] - indexed.loc["baseline", "target_macro_f1"]
            )
            row[f"false_alert_reduction_baseline_minus_{arm_id}"] = float(
                indexed.loc["baseline", "false_alert_rate"] - indexed.loc[arm_id, "false_alert_rate"]
            )
        delta_rows.append(row)
    deltas = pd.DataFrame(delta_rows)
    mining = pd.concat(mining_rows, ignore_index=True) if mining_rows else pd.DataFrame()
    environments = pd.DataFrame(environment_rows)
    return results, summary, split_initialization_summary, deltas, mining, environments


if RUN_REPEATED_GROUP_EVALUATION:
    (
        repeated_seed_results,
        repeated_seed_summary,
        repeated_split_initialization_summary,
        repeated_seed_deltas,
        repeated_seed_hard_negatives,
        repeated_seed_environment_false_alerts,
    ) = run_repeated_group_evaluation()
    repeated_seed_results.to_csv(
        PAPER_DIR / "paper_repeated_seed_results.csv", index=False, encoding="utf-8-sig"
    )
    repeated_seed_summary.to_csv(
        PAPER_DIR / "paper_repeated_seed_summary.csv", index=False, encoding="utf-8-sig"
    )
    repeated_split_initialization_summary.to_csv(
        PAPER_DIR / "paper_repeated_split_initialization_summary.csv", index=False, encoding="utf-8-sig"
    )
    repeated_seed_deltas.to_csv(
        PAPER_DIR / "paper_repeated_seed_deltas.csv", index=False, encoding="utf-8-sig"
    )
    repeated_seed_hard_negatives.to_csv(
        PAPER_DIR / "paper_repeated_seed_hard_negatives.csv", index=False, encoding="utf-8-sig"
    )
    repeated_seed_environment_false_alerts.to_csv(
        PAPER_DIR / "paper_repeated_seed_environment_false_alerts.csv", index=False, encoding="utf-8-sig"
    )
    print("\n5-seed 반복 grouped 평가 요약")
    display(repeated_seed_summary)
    display(repeated_seed_deltas)
else:
    repeated_seed_results = pd.DataFrame()
    repeated_seed_summary = pd.DataFrame()
    repeated_split_initialization_summary = pd.DataFrame()
    repeated_seed_deltas = pd.DataFrame()
    repeated_seed_hard_negatives = pd.DataFrame()
    repeated_seed_environment_false_alerts = pd.DataFrame()


paper_manifest = {
    "pipeline_revision": PIPELINE_REVISION,
    "taxonomy_version": "hearo-8-target-plus-unknown-v2-siren-patterns",
    "run_mode": RUN_MODE,
    "inner_folds": INNER_FOLDS,
    "max_epochs": MAX_EPOCHS,
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "hypothesis": "실제 오류와 확률 기반 semi-hard 선택이 quota 방식보다 F1 손실을 줄이면서 생활소음 오탐을 낮춘다.",
    "selection_protocol": "모든 후보·하드 네거티브·임계값은 development grouped OOF에서 선택",
    "primary_evaluation": "선택 완료 후 untouched grouped test에 동일 초기화 3회로 고정 설정을 적용",
    "seed": SEED,
    "repeated_split_seeds": REPEATED_SPLIT_SEEDS,
    "repeated_group_evaluation_enabled": RUN_REPEATED_GROUP_EVALUATION,
    "final_training_seed_offsets": FINAL_TRAINING_SEED_OFFSETS,
    "representative_final_training_seeds": representative_final_training_seeds,
    "matched_seed_policy": "all four methods share each final-training seed within outer split",
    "test_seed_selection_used": False,
    "deployment_seed_policy": "first pre-registered seed; never best test seed",
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "false_alert_limit": MAX_FALSE_ALERT_RATE,
    "siren_label_definitions": SIREN_LABEL_DEFINITIONS,
    "hard_negative_definition": {
        "source": "development grouped OOF unknown samples only",
        "raw_top1_minimum_confidence": HARD_NEGATIVE_MIN_CONFIDENCE,
        "semihard_minimum_target_probability": SEMI_HARD_MIN_TARGET_PROBABILITY,
        "arms": V3_ARM_SPECS,
        "development_selected_arm": selected_arm_id,
        "candidate_source": "Freesound API v2 CC0-only plus existing development unknowns",
        "quota_minimum_total": MIN_HARD_NEGATIVES_TOTAL,
        "quota_minimum_per_environment_category": MIN_HARD_NEGATIVES_PER_ENV_CATEGORY,
    },
    "versions": {
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "tensorflow_hub": getattr(hub, "__version__", "unknown"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "hardware": {
        "host": socket.gethostname(),
        "platform": platform.platform(),
        "gpu": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "paths": {
        "data_dir": str(DATA_DIR),
        "output_dir": str(OUTPUT_DIR),
        "cache_dir": str(CACHE_DIR),
    },
    "artifacts": sorted(path.name for path in PAPER_DIR.iterdir() if path.is_file()),
}
(PAPER_DIR / "paper_run_manifest.json").write_text(
    json.dumps(json_ready(paper_manifest), ensure_ascii=False, indent=2), encoding="utf-8"
)

protocol_text = """논문 지표 해석 주의사항
1. Development OOF 결과만 방법·임계값·epoch 선택에 사용하며 최종 일반화 성능으로 보고하지 않습니다.
2. 각 outer split에서 네 방법은 동일한 최종 학습 seed 3개를 공유합니다.
3. 대표 split 42의 사전 등록 seed는 1042, 2042, 3042이며 test 성능으로 seed를 선택하지 않습니다.
4. 배포 TFLite는 사전 등록된 첫 seed 1042를 사용하며 가장 좋은 test seed를 사용하지 않습니다.
5. 최종 성능은 방법×outer split×초기화 seed 원시 결과와 paired 차이를 함께 보고합니다.
6. 3개 초기화의 평균·표준편차는 기술통계이며 작은 seed 수로 강한 추론을 하지 않습니다.
7. 95% 신뢰구간은 각 동일 초기화 쌍 안에서 group_id 단위 paired bootstrap으로 계산합니다.
8. 정확 McNemar 검정은 파일 단위 탐색 분석이며 그룹 내 상관 때문에 보조적으로 해석합니다.
9. 반복 outer split의 test 집합은 서로 겹칠 수 있으므로 5-split 평균도 독립 표본 추론으로 과장하지 않습니다.
10. 대표 분할과 반복 분할의 방향이 다르면 방법 효과를 확정하지 않고 초기화·분할 민감성을 명시합니다.

권장 보수적 서술:
“반복 분할에서 준하드 네거티브 방식이 유망한 평균 성능을 보였으나, 대표 분할에서는 동일한 개선이 재현되지 않아 초기화와 데이터 분할에 대한 추가 검증이 필요하다.”
"""
(PAPER_DIR / "paper_reporting_notes.txt").write_text(protocol_text, encoding="utf-8")
conclusion_guardrail = (
    "반복 분할에서 준하드 네거티브 방식이 유망한 평균 성능을 보였으나, "
    "대표 분할에서는 동일한 개선이 재현되지 않아 초기화와 데이터 분할에 대한 "
    "추가 검증이 필요하다."
)
(PAPER_DIR / "paper_conclusion_guardrail.txt").write_text(
    conclusion_guardrail + "\n", encoding="utf-8"
)

# UTF-8/한글 산출물 검증: 대체문자와 빈 파일을 허용하지 않습니다.
for path in sorted(PAPER_DIR.iterdir()):
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"논문 산출물이 비어 있습니다: {path}")
    if path.suffix.lower() in {".csv", ".json", ".tex", ".txt"}:
        decoded = path.read_text(encoding="utf-8-sig")
        if chr(0xFFFD) in decoded:
            raise UnicodeError(f"Unicode replacement character가 발견되었습니다: {path}")

print("\n논문용 주 결과")
display(paper_main_results)
print("\n그룹 paired bootstrap")
display(paper_bootstrap)
print("\n정확 paired 검정")
display(paper_exact_tests)
print("논문 산출물 경로:", PAPER_DIR)


## 12. Google Drive 산출물 확인


In [ ]:
required_outputs = [
    MODEL_DIR / "hearo_classifier_v3_2.tflite",
    MODEL_DIR / "model_metadata_v3_2.json",
    OUTPUT_DIR / "metrics.json",
    OUTPUT_DIR / "hard_negative_candidates_v3.csv",
    OUTPUT_DIR / "hard_negative_arm_membership_v3.csv",
    OUTPUT_DIR / "development_oof_arm_comparison_v3.csv",
    OUTPUT_DIR / "development_oof_arm_metrics_v3.csv",
    PAPER_DIR / "paper_main_results.csv",
    PAPER_DIR / "paper_main_results_by_initialization.csv",
    PAPER_DIR / "paper_reference_seed_results.csv",
    PAPER_DIR / "paper_effect_sizes_by_initialization.csv",
    PAPER_DIR / "paper_grouped_bootstrap_initialization_summary.csv",
    PAPER_DIR / "paper_conclusion_guardrail.txt",
    PAPER_DIR / "paper_environment_false_alerts.csv",
]
if RUN_REPEATED_GROUP_EVALUATION:
    required_outputs.extend([
        PAPER_DIR / "paper_repeated_seed_results.csv",
        PAPER_DIR / "paper_repeated_seed_summary.csv",
        PAPER_DIR / "paper_repeated_seed_deltas.csv",
        PAPER_DIR / "paper_repeated_split_initialization_summary.csv",
        PAPER_DIR / "paper_repeated_seed_hard_negatives.csv",
        PAPER_DIR / "paper_repeated_seed_environment_false_alerts.csv",
    ])
missing_outputs = [str(path) for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing_outputs:
    raise RuntimeError(f"필수 산출물이 없거나 비어 있습니다: {missing_outputs}")
print("Google Drive 저장 완료:", OUTPUT_DIR)
print(f"전체 실행시간: {(time.perf_counter() - PIPELINE_STARTED) / 60:.1f}분")
